# MLstudy: one guided, interactive laboratory

Read, run, change one assumption, then explain the result. This notebook follows **all 22 sidebar sections in order**. It is a downloadable learning artifact: the static website neither executes nor renders it. Every model runs locally; no dataset or model downloads are required after installation.

## Setup and execution
Use a fresh environment with Python **3.11–3.14**. Open this notebook in Jupyter or VS Code and select that environment's Python kernel. Run the installation cell below once to download and install all required packages into the active kernel's environment. Installation requires internet access and changes your local Python environment, not the website. A working Jupyter or VS Code notebook kernel must already be available.

After installation, **restart the kernel**, then run the learning cells below in order. On later visits, skip the installation cell if the packages are already installed.

Use **Restart Kernel → Run All** in order. **For a selected part, run every preceding code cell first, not just the shared imports.** Later parts depend on earlier datasets, models, and validation decisions. Controls display their default experiment immediately and can be changed afterward; a plain printed default is also provided for headless execution. Interactive widgets need a compatible notebook frontend. Later parts deliberately reuse earlier variables; rerun all after changing shared data or preprocessing. Apart from package installation, the example code writes only within an explicitly scoped temporary directory in Part 16 (Jupyter may save the notebook normally).

Optional original synthetic CSVs belong in a sibling `datasets/` directory. Identical deterministic generators below make a standalone download work. No real personal or health information is used; all scores are teaching evidence, not deployment claims.

## Reading this expanded code lesson

Every part contains an explicit representation contract, a comparison hypothesis, numbered operation comments, and a result-reading exercise. Numbering restarts in each cell. Import statements appear at the beginning of the cell that needs them; they reveal dependencies but **do not make sections independent**. Run every preceding learning cell before entering a later part. Shared setup keeps only numerical/tabular basics, display/widgets, version recording and the common control helper.

Creation chooses an estimator recipe, fit learns state, transform reuses feature-preparation state, and predict applies a trained recipe to new rows. Keep these phases distinct throughout. Do not tune final reports after seeing test outcomes.

### Install everything in one step
Run the next cell once. `%pip` installs into this notebook's active Python environment. Restart the kernel afterward, then continue with the route map and learning cells. The example datasets are generated locally, so no separate data download is needed.

In [ ]:
%pip install "numpy>=2.3.4" "pandas>=2.3.3" "scipy>=1.16.3" "scikit-learn>=1.8" "matplotlib>=3.10.7" "ipywidgets>=8.1" "ipykernel>=7.1" "jupyterlab>=4.4" "joblib>=1.5"

## Route map

1. [Part 01 — Introduction and model zoo](#part-01)
2. [Part 02 — The learning lifecycle](#part-02)
3. [Part 03 — Problem framing](#part-03)
4. [Part 04 — Data preparation](#part-04)
5. [Part 05 — Probability and evidence](#part-05)
6. [Part 06 — Decision trees](#part-06)
7. [Part 07 — Linear models and margins](#part-07)
8. [Part 08 — Validation and generalization](#part-08)
9. [Part 09 — Similarity, clustering, and reduction](#part-09)
10. [Part 10 — Evaluation and decisions](#part-10)
11. [Part 11 — Text mining](#part-11)
12. [Part 12 — Associations and ensembles](#part-12)
13. [Part 13 — Neural models and anomaly detection](#part-13)
14. [Part 14 — Optimization](#part-14)
15. [Part 15 — Synthetic PII and PHI handling](#part-15)
16. [Part 16 — Python engineering](#part-16)
17. [Part 17 — Local delivery and monitoring](#part-17)
18. [Part 18 — Teamwork and artifacts](#part-18)
19. [Part 19 — Applied projects](#part-19)
20. [Part 20 — Glossary in practice](#part-20)
21. [Part 21 — Coverage and evidence boundaries](#part-21)
22. [Part 22 — Source notes and reference checks](#part-22)

**Evidence discipline:** fit transformations and choose models on training folds; set policy thresholds on validation only; use the untouched test set once after choices are fixed. Tiny synthetic demonstrations illustrate mechanics, not causal effects or certified performance.

## Shared setup — common basics, not a warehouse of model imports

**Inputs → outputs:** installed packages become NumPy/pandas aliases, display/widget tools, a seed, installed versions and guided_control. Arrays encode numeric shapes; DataFrames preserve field names and row alignment. Specific models, splitters, preprocessing, filesystem tools, plotting and optimization are imported locally later.

The helper first calls an experiment as ordinary Python with scalar widget defaults, so headless execution has a useful output. It then registers interactive callbacks. This is an explicit widgets.interactive call, not an @interact decorator wrapping the function at definition time. A frontend may display the default twice. The callback should report an experiment, not mutate a frozen final policy.

**Call contract:** guided_control(function, parameter_name=widget, ...); every widget keyword must match a function argument. Later lessons still depend on all preceding variables and fits.

In [ ]:
import sys
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

#1 — NumPy supplies numeric arrays; pandas preserves named columns and aligned indexes.
#2 — Widgets/display are frontend tools; they do not learn model parameters or save datasets.
#3 — A fixed seed stabilizes randomized splits and estimators within this installed environment.
SEED = 42
print('Python:', sys.version.split()[0])
#4 — Installed distribution metadata records implementations without contacting a package server.
print({p: metadata.version(p) for p in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'ipywidgets']})

def guided_control(function, **controls):
    #5 — controls maps argument names to widgets; .value extracts each scalar/string default.
    #     ** unpacks that mapping as keywords, so widget names must match function parameters.
    function(**{name: control.value for name, control in controls.items()})
    #6 — interactive registers value-change callbacks and captures their output in a live panel.
    #     This explicit registration is not a decorator; no training policy should mutate here.
    panel = widgets.interactive(function, **controls)
    #7 — Display the panel; the ordinary default call above remains useful without a frontend.
    display(panel)

<a id="part-01"></a>
# Part 01 — Introduction and model zoo

[Read the textbook section](https://juandiegopleitez.com/MLstudy/index.html)

**Learning objective:** Distinguish the installed estimator catalog from a small, purposeful set of demonstrations.

**Step 1.** Query scikit-learn’s public estimator catalog, not a hand-maintained list. Some estimators require special inputs or optional dependencies; listing them is not a claim that every estimator is demonstrated. **Step 2.** Inspect counts and names by task. Later parts compare representative linear, tree, neighbor, kernel, probabilistic, ensemble, neural, clustering, reduction, and anomaly families.

## Guided lesson — Classes, instances, and evidence

**Why this matters:** An estimator class is a configurable recipe; an instance records chosen parameters, and only a fitted instance contains learned state. The public registry tells us what is installed, not what is appropriate or evaluated. Transformer and predictive categories can overlap, so registry counts are not a partition of unique algorithms.

**Expected inputs, outputs, and hypothesis:** Installed scikit-learn classes become a task/name catalog and a bounded browsing view. Before running, predict which families require labels and which learn only from X. Afterward, use a local docstring to identify the required input shape, target type, and fitted output. A catalog entry is not a benchmark recommendation.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** An estimator class is a configurable recipe; an instance records chosen parameters, and only a fitted instance contains learned state. The public registry tells us what is installed, not what is appropriate or evaluated. Transformer and predictive categories can overlap, so registry counts are not a partition of unique algorithms.

**Expected output and reading:** Installed scikit-learn classes become a task/name catalog and a bounded browsing view. Before running, predict which families require labels and which learn only from X. Afterward, use a local docstring to identify the required input shape, target type, and fitted output. A catalog entry is not a benchmark recommendation.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#3 — Registry discovery lists classes without training or evaluating all of them.
from sklearn.utils import all_estimators

#4 — A list of equally keyed dictionaries forms a tidy task/name table, one registry entry per row.
catalog_rows = []
for kind in ['classifier', 'regressor', 'cluster', 'transformer']:
    #5 — type_filter selects estimator capability; discard the class object instead of instantiating
    #     incompatible recipes.
    catalog_rows.extend({'task': kind, 'estimator': name}
                        for name, _ in all_estimators(type_filter=kind))
#6 — Named columns preserve task/name meaning; entries are catalog records, not fitted model results.
catalog = pd.DataFrame(catalog_rows)
#7 — groupby and size count entries per capability; rename/to_frame make the summary a labeled one-column
#     table.
display(catalog.groupby('task').size().rename('installed estimators').to_frame())
#8 — head limits displayed rows without deleting the rest of the catalog or implying a ranking.
display(catalog.head(12))

### Try it — Part 01

Choose a task in the dropdown. Identify two catalog entries you do not know; use their local Python docstrings to describe expected input and output. Which later part demonstrates a representative of that family?

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 01's evidence boundary. Installed scikit-learn classes become a task/name catalog and a bounded browsing view. Before running, predict which families require labels and which learn only from X. Afterward, use a local docstring to identify the required input shape, target type, and fitted output. A catalog entry is not a benchmark recommendation.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** An estimator class is a configurable recipe; an instance records chosen parameters, and only a fitted instance contains learned state. The public registry tells us what is installed, not what is appropriate or evaluated. Transformer and predictive categories can overlap, so registry counts are not a partition of unique algorithms.

**Expected output and reading:** Installed scikit-learn classes become a task/name catalog and a bounded browsing view. Before running, predict which families require labels and which learn only from X. Afterward, use a local docstring to identify the required input shape, target type, and fitted output. A catalog entry is not a benchmark recommendation.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — Widget constructors specify frontend controls; guided_control runs the same default as ordinary Python first.
import ipywidgets as widgets
#3 — Logistic regression learns a regularized linear score and class probabilities.
from sklearn.linear_model import LogisticRegression

#4 — Call browse_catalog(task="regressor") directly; the widget supplies the same named string argument.
def browse_catalog(task='classifier'):
    #5 — eq makes a Boolean mask; loc filters rows, head bounds display, reset_index removes irrelevant
    #     inherited row labels.
    display(catalog.loc[catalog.task.eq(task)].head(20).reset_index(drop=True))
#6 — The helper executes the declared default, then registers callbacks; it returns no replacement final
#     model/policy. Dropdown options constrain valid values and value fixes a reproducible default.
guided_control(browse_catalog, task=widgets.Dropdown(
    options=['classifier', 'regressor', 'cluster', 'transformer'], value='classifier', description='Task'))
#7 — Read the installed class docstring locally; no instance is created and no estimator is trained.
print(LogisticRegression.__doc__.splitlines()[0])

<a id="part-02"></a>
# Part 02 — The learning lifecycle

[Read the textbook section](https://juandiegopleitez.com/MLstudy/lifecycle.html)

**Learning objective:** Turn the lifecycle’s retention pilot into reproducible data and an explicit evidence plan.

**Step 1.** Generate two original, deterministic teaching tables. Customers represent a retention snapshot; sensors represent ordered equipment observations. Arithmetic formulas intentionally replace random-state or cross-language ambiguity. **Step 2.** Load optional CSVs only if present and verify their schemas and values against the generators. No files are created here.

## Guided lesson — Data contracts come before algorithms

**Why this matters:** The retention table is invented to provide some learnable signal, overlapping classes, bounded fields, and visible missingness. Sensor rows represent ordered observations with deliberately injected outliers. None are real customers, diagnoses, or equipment measurements. Deterministic arithmetic avoids network access and random-generator-version differences.

**Expected inputs, outputs, and hypothesis:** Zero-argument generator calls produce 240 customer rows with six columns and 180 sensor rows with five columns. Optional local CSVs must match values, names, order and missingness. A missing CSV chooses the identical fallback; a present but changed CSV should fail loudly. Distinguish a successful schema check from evidence that a dataset represents a population.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Generate customer rows: features, identifiers, missingness, and a fabricated target. Predict the signal direction before inspecting any classifier.

In [ ]:
#1 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import numpy as np
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#3 — No arguments are needed; return a fresh six-column DataFrame with identical values on every call.
def synthetic_customers():
    #4 — A list accumulates fixed-order row tuples; explicit DataFrame columns later define what each tuple
    #     position means.
    rows = []
    for i in range(240):
        #5 — Modular arithmetic bounds tenure to 1..60 months; this is designed teaching signal, not
        #     population sampling.
        tenure = (i * 7) % 60 + 1
        #6 — A different bounded arithmetic cycle keeps spend numeric and avoids making all columns copies
        #     of the same pattern.
        spend = 30 + (i * 13) % 100
        #7 — Integer counts range 0..6; their relationship with the later label is intentionally learnable.
        tickets = (i * 11) % 7
        #8 — Weighted conditions add risk and deterministic jitter creates class overlap; int converts the
        #     resulting Boolean to 0/1.
        churn = int(2 * tickets + 4 * (tenure < 12) + 2 * (spend > 100)
                    + (i * 17) % 11 - 5 > 9)
        #9 — SYN IDs are placeholders, not features; np.nan every nineteenth row creates a deliberate
        #     imputation need.
        rows.append((f'SYN-{i:03d}', np.nan if i % 19 == 0 else tenure,
                     spend, tickets, ['web', 'store', 'phone'][i % 3], churn))
    #10 — Six names align with tuple order; missing tenure becomes a floating column under ordinary pandas
    #     inference.
    return pd.DataFrame(rows, columns=['record_id', 'tenure_months', 'monthly_spend',
                                       'support_tickets', 'channel', 'churn'])

### Part 02 — guided step 2

Generate ordered sensor observations with known injected outliers. This label-informed setting is deliberately easier than an unlabeled operational stream.

In [ ]:
#1 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#2 — Return ordered readings with a fabricated anomaly target; keep order and labels out of detector
#     features.
def synthetic_sensors():
    #3 — A list accumulates fixed-order row tuples; explicit DataFrame columns later define what each tuple
    #     position means.
    rows = []
    for i in range(180):
        #4 — Two periodic conditions inject easy labeled anomalies; this known mechanism limits realism.
        unusual = int(i % 23 == 0 or i % 31 == 0)
        #5 — Each outlier shifts three measured fields while the first field preserves reading order.
        rows.append((i, 20 + (i * 7) % 9 + 16 * unusual,
                     2 + (i * 3) % 5 + 9 * unusual,
                     95 + (i * 11) % 13 - 20 * unusual, unusual))
    #6 — The five-field schema ends with evaluation metadata; using is_anomaly as an input would reveal the
    #     answer.
    return pd.DataFrame(rows, columns=['reading_index', 'temperature', 'vibration',
                                       'pressure', 'is_anomaly'])

### Part 02 — guided step 3

Define optional_table(filename, expected_frame). A relative datasets path is read only if it exists; a present mismatching file should raise rather than silently change the experiment.

In [ ]:
#1 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#2 — Path composes local locations without creating a file or accessing a network.
from pathlib import Path

#3 — Accept a CSV basename and expected reference frame; return a verified loaded frame or that in-memory
#     reference.
def optional_table(filename, expected):
    #4 — Path joining expresses a relative local dataset location; missing files cause fallback, not a
    #     download or write.
    path = Path('datasets') / filename
    #5 — Check file existence before reading; absent optional companions are an expected standalone-download
    #     mode.
    if path.is_file():
        #6 — Local CSV parsing infers dtypes and blanks as missing values; schema/value assertions below
        #     check the result.
        actual = pd.read_csv(path)
        #7 — Compare column/index order, values and NaNs; check_dtype=False tolerates equivalent CSV
        #     inference, not changed observations.
        pd.testing.assert_frame_equal(actual, expected, check_dtype=False)
        print(f'{filename}: CSV matches deterministic fallback')
        return actual
    print(f'{filename}: using identical in-memory fallback')
    return expected

### Part 02 — guided step 4

Load both frames and inspect five rows. Expect CSV-match or fallback messages followed by named tables; neither path creates data files.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display

#2 — The generator argument is evaluated even if the CSV exists, supplying a deterministic reference for
#     comparison.
customers = optional_table('study-customers.csv', synthetic_customers())
#3 — A separate schema prevents conflating chronological sensor records with independent customer snapshots.
sensors = optional_table('study-sensors.csv', synthetic_sensors())
#4 — Preview five records from each table to inspect representation; head is not a training/validation split.
display(customers.head(), sensors.head())

### Try it — Part 02

Inspect missingness and class balance. Add a concrete acceptance criterion to the lifecycle plan: include a metric, a decision owner, and a condition that would stop the pilot. This notebook cannot establish whether contacting a customer changes retention.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 02's evidence boundary. Zero-argument generator calls produce 240 customer rows with six columns and 180 sensor rows with five columns. Optional local CSVs must match values, names, order and missingness. A missing CSV chooses the identical fallback; a present but changed CSV should fail loudly. Distinguish a successful schema check from evidence that a dataset represents a population.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** The retention table is invented to provide some learnable signal, overlapping classes, bounded fields, and visible missingness. Sensor rows represent ordered observations with deliberately injected outliers. None are real customers, diagnoses, or equipment measurements. Deterministic arithmetic avoids network access and random-generator-version differences.

**Expected output and reading:** Zero-argument generator calls produce 240 customer rows with six columns and 180 sensor rows with five columns. Optional local CSVs must match values, names, order and missingness. A missing CSV chooses the identical fallback; a present but changed CSV should fail loudly. Distinguish a successful schema check from evidence that a dataset represents a population.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#3 — isna marks missing cells and columnwise sum counts them, revealing where cleaning is needed.
display(customers.isna().sum().rename('missing').to_frame())
#4 — normalize=True converts label counts to fractions; imbalance changes how a dummy baseline should be
#     interpreted.
display(customers.churn.value_counts(normalize=True).rename('fraction').to_frame())
#5 — Each row couples a lifecycle stage to an accountable role and a checkable evidence artifact.
lifecycle_plan = pd.DataFrame([
    ('Frame', 'retention lead', 'define eligible decisions and contact costs'),
    ('Prepare', 'data steward', 'verify point-in-time availability'),
    ('Model', 'analyst', 'beat a dummy baseline on training CV'),
    ('Evaluate', 'reviewer', 'freeze threshold before final test'),
    ('Deliver', 'operator', 'approve local input contract and rollback'),
    ('Monitor', 'owner', 'investigate missingness or outcome drift'),
], columns=['stage', 'owner', 'acceptance evidence'])
#6 — Render this small named table locally; display does not save a file, fetch data, or establish external
#     approval.
display(lifecycle_plan)

<a id="part-03"></a>
# Part 03 — Problem framing

[Read the textbook section](https://juandiegopleitez.com/MLstudy/problem-framing.html)

**Learning objective:** Write a prediction contract and separate risk prediction from intervention value.

**Step 1.** Define the unit, prediction time, horizon, and available fields. Synthetic churn is a fabricated label, not a causal outcome. **Step 2.** Exclude record identifiers and create stratified 60/20/20 training, validation, and test partitions before any learned preprocessing. These independent synthetic rows justify random splitting; real repeated customers or temporal data require group/time splits.

## Guided lesson — Specify what can be known at prediction time

**Why this matters:** A prediction contract fixes the unit, horizon, information availability and decision. Churn risk and outreach benefit are different estimands: a high-risk person may not respond to outreach. Record identifiers are traceability metadata, not predictors. Random splitting is justified only by the independent synthetic-row assumption.

**Expected inputs, outputs, and hypothesis:** A feature DataFrame X has shape (n,4); a target Series y has shape (n,). A 40% holdout split followed by a half split yields 60/20/20 training/validation/test proportions. Training learns state, validation chooses policies, and the final test is opened once later. Real repeated customers require grouping, and future decisions require chronology.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** A prediction contract fixes the unit, horizon, information availability and decision. Churn risk and outreach benefit are different estimands: a high-risk person may not respond to outreach. Record identifiers are traceability metadata, not predictors. Random splitting is justified only by the independent synthetic-row assumption.

**Expected output and reading:** A feature DataFrame X has shape (n,4); a target Series y has shape (n,). A 40% holdout split followed by a half split yields 60/20/20 training/validation/test proportions. Training learns state, validation chooses policies, and the final test is opened once later. Real repeated customers require grouping, and future decisions require chronology.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Split aligned features/labels before learning representation statistics.
from sklearn.model_selection import train_test_split

#2 — A three-name list defines the numeric schema; the companion category list keeps string handling
#     explicit.
numeric_features = ['tenure_months', 'monthly_spend', 'support_tickets']
categorical_features = ['channel']
feature_columns = numeric_features + categorical_features
#3 — A dictionary documents unit, time, horizon, exclusions and decision; this policy text is not passed to
#     an estimator.
contract = {'unit': 'synthetic customer snapshot', 'time': 'before outreach',
            'horizon': 'illustrative next 30 days', 'target': 'churn',
            'excluded': ['record_id'], 'decision': 'review for an optional retention pilot'}
#4 — Column selection yields an (n_rows,4) DataFrame without target or ID; names survive preprocessing
#     routing.
X = customers[feature_columns]
#5 — The one-dimensional label Series retains the same row index so splitting and metrics remain aligned.
y = customers.churn
#6 — test_size=0.4 reserves 40%; stratify maintains approximate label balance; random_state fixes the
#     randomized partition.
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=SEED)
#7 — Halving the 40% holdout yields 20% validation and 20% final test, neither used to learn preprocessing.
X_val, X_test, y_val, y_test = train_test_split(
    X_holdout, y_holdout, test_size=0.5, stratify=y_holdout, random_state=SEED)
#8 — Index-set tests establish non-overlapping rows; they cannot prove independence if real entities repeat
#     with different row IDs.
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
print(contract)
print({'train': len(X_train), 'validation': len(X_val), 'test': len(X_test)})

### Try it — Part 03

Change the assumed contact cost and success benefit below. Explain why a high churn probability alone cannot identify treatment benefit: an experiment or defensible causal design is still needed.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 03's evidence boundary. A feature DataFrame X has shape (n,4); a target Series y has shape (n,). A 40% holdout split followed by a half split yields 60/20/20 training/validation/test proportions. Training learns state, validation chooses policies, and the final test is opened once later. Real repeated customers require grouping, and future decisions require chronology.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** A prediction contract fixes the unit, horizon, information availability and decision. Churn risk and outreach benefit are different estimands: a high-risk person may not respond to outreach. Record identifiers are traceability metadata, not predictors. Random splitting is justified only by the independent synthetic-row assumption.

**Expected output and reading:** A feature DataFrame X has shape (n,4); a target Series y has shape (n,). A 40% holdout split followed by a half split yields 60/20/20 training/validation/test proportions. Training learns state, validation chooses policies, and the final test is opened once later. Real repeated customers require grouping, and future decisions require chronology.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import numpy as np
#3 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#4 — A shape-(3,) float array contains hypothetical causal effects, not estimated churn probabilities.
assumed_incremental_effect = np.array([0.01, 0.08, 0.15])
contact_cost, retained_value = 5.0, 100.0
#5 — Scalar broadcasting applies value*effect-cost to every scenario; a negative number rejects that assumed
#     intervention economics.
planning_values = retained_value * assumed_incremental_effect - contact_cost
#6 — Equal-length scenario vectors form paired rows, not independent estimates of treatment response.
display(pd.DataFrame({'assumed_effect_not_estimated': assumed_incremental_effect,
                      'hypothetical_net_value': planning_values}))

<a id="part-04"></a>
# Part 04 — Data preparation

[Read the textbook section](https://juandiegopleitez.com/MLstudy/data-preparation.html)

**Learning objective:** Fit missing-value handling, encoding, and scaling inside the training boundary.

**Step 1.** Use medians and missingness indicators for numeric fields, then standardize. Encode categories with an explicit unseen-category policy. **Step 2.** Build a reusable unfitted preprocessor and clone it into each model pipeline. Validation and test rows never determine medians, scales, or categories.

## Guided lesson — Cleaning is learned state, not just housekeeping

**Why this matters:** Strings and missing numeric values are not valid numeric matrix inputs for many estimators. Median imputation prevents NaN errors; encoding converts categories to numeric features; scaling prevents large-unit features dominating distance or regularization. Their replacement values, vocabulary, means and scales must come only from training data.

**Expected inputs, outputs, and hypothesis:** A raw mixed-type frame becomes a numeric matrix through separate branches, then a logistic model. fit learns preprocessing and coefficients; transform applies already learned preprocessing without updating it; predict transforms and returns labels. Hypothesis: a pipeline remains computationally robust to missing values and unseen categories, although operational validation may still reject unknown categories.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Build an unfitted ColumnTransformer and customer_model(estimator) factory. The factory accepts a model instance and returns a fresh raw-frame-to-prediction pipeline; creating the recipe learns no statistics.

In [ ]:
#1 — clone copies configuration while discarding learned state, preventing fitted-state leakage.
from sklearn.base import clone
#2 — ColumnTransformer routes named columns through separate numeric/category branches.
from sklearn.compose import ColumnTransformer
#3 — Pipeline keeps feature preparation inside every training and cross-validation boundary.
from sklearn.pipeline import Pipeline
#4 — make_pipeline derives lowercase step names and chains fit/transform/predict responsibilities.
from sklearn.pipeline import make_pipeline
#5 — SimpleImputer learns replacements from fit data so missing values do not break estimators.
from sklearn.impute import SimpleImputer
#6 — StandardScaler learns feature means/scales; transform reuses rather than updates that state.
from sklearn.preprocessing import StandardScaler
#7 — OneHotEncoder turns learned categories into fixed-width numeric feature columns.
from sklearn.preprocessing import OneHotEncoder

#8 — Named branches receive selected columns and concatenate numeric outputs in declared branch order.
preprocessor = ColumnTransformer([
    #9 — Median replacement resists extremes; add_indicator creates missingness flags; scaling then balances
    #     feature magnitudes.
    ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True),
                              StandardScaler()), numeric_features),
    #10 — Most-frequent imputation prevents missing-category failures before categorical encoding.
    ('category', make_pipeline(SimpleImputer(strategy='most_frequent'),
                               #11 — Unknown values map to all-zero category columns rather than errors;
                               #     sparse_output=False keeps this small demo dense.
                               OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
     categorical_features),
])
#12 — Accept an estimator instance and return a fresh Pipeline from raw feature DataFrames to the estimator
#     output.
def customer_model(estimator):
    #13 — clone discards any learned preprocessor state; named steps expose inspection while fit learns only
    #     inside its supplied rows.
    return Pipeline([('prepare', clone(preprocessor)), ('model', estimator)])

### Part 04 — guided step 2

Create the logistic recipe, then fit it. Expect an expanded numeric feature count and a validation balanced-accuracy value; transform inspects learned state without learning from validation.

In [ ]:
#1 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics
#2 — Logistic regression learns a regularized linear score and class probabilities.
from sklearn.linear_model import LogisticRegression

#3 — Instantiation chooses the recipe; max_iter=1000 permits convergence and random_state records any solver
#     randomness.
logistic = customer_model(LogisticRegression(max_iter=1000, random_state=SEED))
#4 — fit first learns medians/scales/categories, then coefficients from transformed training rows; creation
#     alone learned nothing.
logistic.fit(X_train, y_train)
#5 — named_steps accesses fitted preparation; transform reuses state and shape reports (rows, expanded
#     numeric features).
print('Transformed training shape:', logistic.named_steps['prepare'].transform(X_train).shape)
#6 — predict yields one hard label per row; balanced accuracy averages class-specific recalls instead of
#     favoring the majority.
print('Validation balanced accuracy:', round(metrics.balanced_accuracy_score(
    y_val, logistic.predict(X_val)), 3))

### Try it — Part 04

Create an unseen category and a missing tenure. Verify prediction remains finite. Then consider whether silently accepting an unknown category is suitable for a production contract; successful computation does not guarantee valid semantics.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 04's evidence boundary. A raw mixed-type frame becomes a numeric matrix through separate branches, then a logistic model. fit learns preprocessing and coefficients; transform applies already learned preprocessing without updating it; predict transforms and returns labels. Hypothesis: a pipeline remains computationally robust to missing values and unseen categories, although operational validation may still reject unknown categories.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Strings and missing numeric values are not valid numeric matrix inputs for many estimators. Median imputation prevents NaN errors; encoding converts categories to numeric features; scaling prevents large-unit features dominating distance or regularization. Their replacement values, vocabulary, means and scales must come only from training data.

**Expected output and reading:** A raw mixed-type frame becomes a numeric matrix through separate branches, then a logistic model. fit learns preprocessing and coefficients; transform applies already learned preprocessing without updating it; predict transforms and returns labels. Hypothesis: a pipeline remains computationally robust to missing values and unseen categories, although operational validation may still reject unknown categories.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import numpy as np

#2 — iloc takes two rows by position; copy prevents probing robustness from changing original validation
#     data.
unseen_example = X_val.iloc[:2].copy()
#3 — loc uses preserved row labels to replace a category; the trained encoder has never learned this string.
unseen_example.loc[unseen_example.index[0], 'channel'] = 'new-channel'
#4 — NaN probes the training-derived numeric replacement without refitting it.
unseen_example.loc[unseen_example.index[1], 'tenure_months'] = np.nan
#5 — predict_proba returns (2,n_classes); [:,1] extracts churn probability because these labels are ordered
#     [0,1].
unseen_probabilities = logistic.predict_proba(unseen_example)[:, 1]
#6 — Elementwise finite checks reject NaN/infinity; all reduces the Boolean vector to one contract assertion.
assert np.isfinite(unseen_probabilities).all()
print('Unknown-category probabilities:', unseen_probabilities.round(3))
#7 — named_transformers_ and statistics_ expose learned state, unlike unfitted constructor parameters.
print('Training medians:', logistic.named_steps['prepare'].named_transformers_[
    'numeric'].named_steps['simpleimputer'].statistics_)

<a id="part-05"></a>
# Part 05 — Probability and evidence

[Read the textbook section](https://juandiegopleitez.com/MLstudy/probability.html)

**Learning objective:** Use Bayes’ rule to reason about rare alerts and compare a probabilistic classifier.

**Step 1.** Translate sensitivity, specificity, and prevalence into positive predictive value. A seemingly accurate alert may still have low precision when the event is rare. **Step 2.** Move only prevalence and observe the posterior; this is arithmetic, not a learned population estimate.

## Guided lesson — Conditioning changes the question

**Why this matters:** Sensitivity P(alert | event) is not positive predictive value P(event | alert). Bayes combines sensitivity, specificity and prevalence by accounting for all ways an alert occurs. The slider assumes fixed test properties across populations; that simplification may fail in reality.

**Expected inputs, outputs, and hypothesis:** A scalar prevalence produces a posterior probability; a separate Gaussian naive Bayes pipeline produces validation predictions. Hypothesis: rare events have many false alerts even with high sensitivity/specificity. Compare the classifier with logistic on identical rows, but do not infer calibration from a good classification score.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Sensitivity P(alert | event) is not positive predictive value P(event | alert). Bayes combines sensitivity, specificity and prevalence by accounting for all ways an alert occurs. The slider assumes fixed test properties across populations; that simplification may fail in reality.

**Expected output and reading:** A scalar prevalence produces a posterior probability; a separate Gaussian naive Bayes pipeline produces validation predictions. Hypothesis: rare events have many false alerts even with high sensitivity/specificity. Compare the classifier with logistic on identical rows, but do not infer calibration from a good classification score.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Widget constructors specify frontend controls; guided_control runs the same default as ordinary Python first.
import ipywidgets as widgets

#2 — The argument is a fraction such as 0.01, not a percentage number 1; the helper reports arithmetic
#     without fitting data.
def rare_alert(prevalence=0.01):
    sensitivity, specificity = 0.9, 0.95
    #3 — Total alert probability sums true-alert and false-alert contributions weighted by event prevalence
    #     and its complement.
    evidence = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
    #4 — Bayes divides the event-alert joint probability by all alerts, conditioning on the observation
    #     rather than the unknown event.
    posterior = sensitivity * prevalence / evidence
    print(f'Prior={prevalence:.1%}; P(event | alert)={posterior:.1%}')
#5 — The helper executes the declared default, then registers callbacks; it returns no replacement final
#     model/policy. FloatSlider min/max/step bound the scenario and description labels it; readout_format
#     affects display only.
guided_control(rare_alert, prevalence=widgets.FloatSlider(
    min=0.001, max=0.2, step=0.001, value=0.01, description='Prior', readout_format='.3f'))

### Try it — Part 05

Run Gaussian naive Bayes on the same training-only numeric fields. Compare validation balanced accuracy with Part 4. Explain why correlated fields can violate the conditional-independence assumption and why the probability should not be trusted without calibration evidence.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 05's evidence boundary. A scalar prevalence produces a posterior probability; a separate Gaussian naive Bayes pipeline produces validation predictions. Hypothesis: rare events have many false alerts even with high sensitivity/specificity. Compare the classifier with logistic on identical rows, but do not infer calibration from a good classification score.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Sensitivity P(alert | event) is not positive predictive value P(event | alert). Bayes combines sensitivity, specificity and prevalence by accounting for all ways an alert occurs. The slider assumes fixed test properties across populations; that simplification may fail in reality.

**Expected output and reading:** A scalar prevalence produces a posterior probability; a separate Gaussian naive Bayes pipeline produces validation predictions. Hypothesis: rare events have many false alerts even with high sensitivity/specificity. Compare the classifier with logistic on identical rows, but do not infer calibration from a good classification score.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics
#2 — Gaussian NB learns per-class feature distributions and class priors.
from sklearn.naive_bayes import GaussianNB

#3 — Create GaussianNB, wrap in fresh preprocessing, then fit to learn class priors and feature
#     distributions inside training only.
naive_bayes = customer_model(GaussianNB()).fit(X_train, y_train)
#4 — Score held-out hard labels; a good score does not verify that predicted probabilities are calibrated.
print('Naive Bayes validation balanced accuracy:', round(metrics.balanced_accuracy_score(
    y_val, naive_bayes.predict(X_val)), 3))

<a id="part-06"></a>
# Part 06 — Decision trees

[Read the textbook section](https://juandiegopleitez.com/MLstudy/decision-trees.html)

**Learning objective:** Connect entropy, tree depth, leaf probabilities, and regression impurity.

**Step 1.** Compute binary entropy at a balanced parent. **Step 2.** Fit a depth-controlled classification tree using training rows only; inspect validation scores and the learned rules. Depth changes are exploratory validation choices, never reasons to inspect test outcomes.

## Guided lesson — Trees turn uncertainty into local rules

**Why this matters:** Classification trees choose feature thresholds to reduce class impurity and predict leaf class proportions. Numeric targets require residual-based impurity instead. More leaves can memorize small regions; max_depth and min_samples_leaf limit that capacity. Exported rules describe transformed features, not necessarily raw units.

**Expected inputs, outputs, and hypothesis:** A probability vector becomes scalar entropy in bits; training data becomes readable rules and train/validation scores; a one-feature curve becomes numeric leaf predictions. Hypothesis: increasing depth improves training score more reliably than validation score. Read the gap, not just the largest printed number.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Compute entropy on known probability vectors. Check inputs are legitimate probabilities; filtering zero mass implements a limiting convention rather than hiding arbitrary invalid inputs.

In [ ]:
#1 — Local tree dependencies are declared before entropy arithmetic so the learning objective and later rules remain connected.
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import numpy as np

#3 — Accept a one-dimensional probability sequence assumed nonnegative and summing to one; return a scalar
#     in bits.
def entropy(probabilities):
    #4 — Convert list/array input to a numeric floating vector so logarithms and multiplication have a clear
    #     dtype.
    p = np.asarray(probabilities, dtype=float)
    #5 — Boolean indexing omits zero-mass terms, implementing the limiting 0*log(0)=0 convention and
    #     avoiding nonfinite arithmetic.
    p = p[p > 0]
    #6 — log2 measures bits; multiply by mass, sum across classes, negate, then return an ordinary scalar.
    return float(-(p * np.log2(p)).sum())
print('Balanced parent entropy:', entropy([0.5, 0.5]))
#7 — Weight child entropies by child population fractions; this equal-size example is a calculation, not a
#     fitted tree.
print('Weighted child entropy:', 0.5 * entropy([0.9, 0.1]) + 0.5 * entropy([0.1, 0.9]))

### Part 06 — guided step 2

Train a new tree for each depth and compare train/validation scores. Rule text is deliberately truncated; deeper fitted trees may contain more rules than printed.

In [ ]:
#1 — Widget constructors specify frontend controls; guided_control runs the same default as ordinary Python first.
import ipywidgets as widgets
#2 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics
#3 — Classification trees learn split rules and leaf class proportions.
from sklearn.tree import DecisionTreeClassifier
#4 — Rule export needs feature names in the transformed matrix order.
from sklearn.tree import export_text

#5 — Accept a maximum depth and construct a fresh model for that validation scenario; do not mutate the
#     final logistic baseline.
def tree_experiment(depth=3):
    #6 — max_depth limits nesting and min_samples_leaf=5 reduces tiny leaves; fixed seed stabilizes split
    #     ties before fit.
    model = customer_model(DecisionTreeClassifier(
        max_depth=depth, min_samples_leaf=5, random_state=SEED)).fit(X_train, y_train)
    #7 — The same fitted tree scores train and validation separately; a widening gap suggests memorization
    #     rather than useful capacity.
    print({split: round(metrics.balanced_accuracy_score(labels, model.predict(data)), 3)
           for split, data, labels in [('train', X_train, y_train), ('validation', X_val, y_val)]})
    #8 — Use transformed feature names in matrix order; max_depth=2 truncates only exported text, not the
    #     fitted tree.
    print(export_text(model.named_steps['model'],
                      feature_names=list(model.named_steps['prepare'].get_feature_names_out()),
                      max_depth=2))
#9 — The helper executes the declared default, then registers callbacks; it returns no replacement final
#     model/policy. IntSlider uses discrete values; odd k steps avoid most binary vote ties.
guided_control(tree_experiment, depth=widgets.IntSlider(min=1, max=8, value=3, description='Depth'))

### Try it — Part 06

Change depth and note whether training gains also improve validation. Fit a shallow regression tree to a generated numeric target; explain why squared-error impurity is not classification entropy.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 06's evidence boundary. A probability vector becomes scalar entropy in bits; training data becomes readable rules and train/validation scores; a one-feature curve becomes numeric leaf predictions. Hypothesis: increasing depth improves training score more reliably than validation score. Read the gap, not just the largest printed number.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Classification trees choose feature thresholds to reduce class impurity and predict leaf class proportions. Numeric targets require residual-based impurity instead. More leaves can memorize small regions; max_depth and min_samples_leaf limit that capacity. Exported rules describe transformed features, not necessarily raw units.

**Expected output and reading:** A probability vector becomes scalar entropy in bits; training data becomes readable rules and train/validation scores; a one-feature curve becomes numeric leaf predictions. Hypothesis: increasing depth improves training score more reliably than validation score. Read the gap, not just the largest printed number.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — NumPy expresses the numeric arrays, elementwise operations and explicit shape contract in this cell.
import numpy as np
#2 — Regression trees learn numeric leaf means and residual-based splits.
from sklearn.tree import DecisionTreeRegressor

#3 — linspace creates 80 evenly spaced inputs; reshape(-1,1) forms the required (80 samples,1 feature)
#     matrix.
x_curve = np.linspace(-2, 2, 80).reshape(-1, 1)
#4 — [:,0] extracts a vector; elementwise square and sine create a one-dimensional numeric target, not extra
#     input features.
y_curve = x_curve[:, 0] ** 2 + 0.1 * np.sin(8 * x_curve[:, 0])
#5 — A depth-3 regression recipe is created then fitted; default squared-error impurity learns numeric leaf
#     means.
regression_tree = DecisionTreeRegressor(max_depth=3, random_state=SEED).fit(x_curve, y_curve)
#6 — Inspect fitted leaf count as a complexity measure, not as a performance estimate.
print('Illustrative regression leaves:', regression_tree.get_n_leaves())
#7 — Nested lists form (3,1) inputs; predict returns three numeric values, unlike classification
#     probabilities.
print('Predictions at -1, 0, 1:', regression_tree.predict([[-1], [0], [1]]).round(3))

<a id="part-07"></a>
# Part 07 — Linear models and margins

[Read the textbook section](https://juandiegopleitez.com/MLstudy/linear-models.html)

**Learning objective:** Compare linear, regularized, and margin-based models with appropriate losses.

**Step 1.** Compare logistic regression and a scaled support-vector classifier against a dummy classifier on training cross-validation. Accuracy alone can hide class imbalance. **Step 2.** Reserve an independent test partition of the built-in diabetes regression data, and compare regressors with training CV mean absolute error. The medical dataset is a packaged teaching benchmark, not clinical advice.

## Guided lesson — Compare hypotheses under one evaluation design

**Why this matters:** Logistic regression is a regularized linear baseline; an RBF support-vector classifier allows nonlinear boundaries but depends on scaled distances and C. Dummy predictors ignore informative features and reveal whether a learned model adds value. Regression requires numeric losses rather than classification accuracy.

**Expected inputs, outputs, and hypothesis:** Identical stratified customer folds produce classifier scores. A separate packaged diabetes dataset creates regression arrays, a reserved test set, and training-CV MAE comparisons. Hypothesis: nonlinear flexibility may help, but a simpler regularized model may be more stable. MAE is in target units; negative sklearn scoring must be converted back before interpretation.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Compare dummy, logistic and SVC on identical stratified folds. The hypothesis is that learned signal beats prevalence, while extra nonlinear capacity must earn its complexity.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#3 — Stratified folds approximately preserve class proportions while excluding scored rows.
from sklearn.model_selection import StratifiedKFold
#4 — CV clones and fits the complete recipe separately inside each training fold.
from sklearn.model_selection import cross_validate
#5 — A feature-ignoring class baseline tests whether informative modeling adds value.
from sklearn.dummy import DummyClassifier
#6 — Logistic regression learns a regularized linear score and class probabilities.
from sklearn.linear_model import LogisticRegression
#7 — The default RBF classifier uses scaled distances and a soft-margin penalty.
from sklearn.svm import SVC

#8 — Four stratified shuffled folds stabilize class balance; the seed reuses an identical split plan across
#     candidate models.
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
#9 — Report names map to unfitted full pipelines; a prior-only dummy tests value beyond class proportions.
classifier_candidates = {
    'dummy': customer_model(DummyClassifier(strategy='prior')),
    'logistic': customer_model(LogisticRegression(max_iter=1000, random_state=SEED)),
    #10 — C=1 controls margin violations; default RBF distances depend on the training-only standardized
    #     representation.
    'SVC': customer_model(SVC(C=1.0)),
}
classifier_cv = []
for name, model in classifier_candidates.items():
    #11 — Fit the entire recipe independently inside each fold; test_score denotes CV exclusions, never the
    #     reserved final X_test.
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring='balanced_accuracy')
    #12 — Mean summarizes fold balanced accuracy; training-CV ranking is development evidence, not a final
    #     deployment claim.
    classifier_cv.append({'model': name, 'CV balanced accuracy': scores['test_score'].mean()})
#13 — Convert equally keyed rows or aligned vectors into a named table so comparisons retain their meaning
#     and units.
display(pd.DataFrame(classifier_cv))

### Part 07 — guided step 2

Load a separate local continuous-target benchmark and reserve its final test. Define four numeric recipes; do not inspect test errors during recipe selection.

In [ ]:
#1 — These packaged datasets are installed locally, not fetched during execution.
from sklearn import datasets
#2 — make_pipeline derives lowercase step names and chains fit/transform/predict responsibilities.
from sklearn.pipeline import make_pipeline
#3 — StandardScaler learns feature means/scales; transform reuses rather than updates that state.
from sklearn.preprocessing import StandardScaler
#4 — Split aligned features/labels before learning representation statistics.
from sklearn.model_selection import train_test_split
#5 — A mean-target baseline tests whether numeric models outperform a constant prediction.
from sklearn.dummy import DummyRegressor
#6 — Linear regression minimizes squared numeric residuals without Ridge shrinkage.
from sklearn.linear_model import LinearRegression
#7 — Ridge adds an L2 coefficient penalty, trading training fit for stability.
from sklearn.linear_model import Ridge
#8 — SVR predicts numeric targets with an epsilon-insensitive loss and a distance-based kernel.
from sklearn.svm import SVR

#9 — return_X_y=True yields a numeric (n,10) feature array and (n,) target vector from installed local
#     benchmark data.
X_diabetes, y_diabetes = datasets.load_diabetes(return_X_y=True)
#10 — Reserve a separate 25% regression test before comparison; continuous targets are not passed to class
#     stratification.
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    #11 — return_X_y=True yields a numeric (n,10) feature array and (n,) target vector from installed local
    #     benchmark data.
    X_diabetes, y_diabetes, test_size=0.25, random_state=SEED)
#12 — Compare a target-mean dummy, unpenalized linear fit, L2 shrinkage, and a nonlinear kernel under one
#     numeric task.
regressors = {'dummy': DummyRegressor(), 'linear': LinearRegression(),
              #13 — StandardScaler lives inside Ridge pipeline; alpha=10 is a declared L2 penalty, not a
              #     learned coefficient.
              'ridge': make_pipeline(StandardScaler(), Ridge(alpha=10)),
              #14 — C=50 penalizes errors outside the epsilon tube; scaling remains inside each CV training
              #     fold.
              'SVR': make_pipeline(StandardScaler(), SVR(C=50))}

### Part 07 — guided step 3

Compute training-CV MAE, convert negative scorers to positive error, and inspect fold spread. Lower MAE is better; Part 10 alone opens the regression final test.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#3 — CV clones and fits the complete recipe separately inside each training fold.
from sklearn.model_selection import cross_validate

regression_cv = []
for name, model in regressors.items():
    #4 — cv=4 uses regression folds; neg_mean_absolute_error follows sklearn larger-is-better convention, so
    #     scores are negative MAE.
    values = cross_validate(model, X_reg_train, y_reg_train, cv=4,
                            scoring='neg_mean_absolute_error')['test_score']
    #5 — Negate the mean to restore positive MAE; std describes fold-to-fold variation rather than an
    #     independent confidence interval.
    regression_cv.append({'model': name, 'CV MAE': -values.mean(), 'fold SD': values.std()})
#6 — Sort by lower training-CV MAE before looking at final test outcomes; the first row defines the recipe
#     later tested.
regression_results = pd.DataFrame(regression_cv).sort_values('CV MAE')
#7 — Render this small named table locally; display does not save a file, fetch data, or establish external
#     approval.
display(regression_results)

### Try it — Part 07

Change regularization strength below. Compare mean validation performance and fold spread, not coefficient size alone. Select any alternative before the final test in Part 10; standardizing a feature changes the interpretation of its coefficient.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 07's evidence boundary. Identical stratified customer folds produce classifier scores. A separate packaged diabetes dataset creates regression arrays, a reserved test set, and training-CV MAE comparisons. Hypothesis: nonlinear flexibility may help, but a simpler regularized model may be more stable. MAE is in target units; negative sklearn scoring must be converted back before interpretation.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Logistic regression is a regularized linear baseline; an RBF support-vector classifier allows nonlinear boundaries but depends on scaled distances and C. Dummy predictors ignore informative features and reveal whether a learned model adds value. Regression requires numeric losses rather than classification accuracy.

**Expected output and reading:** Identical stratified customer folds produce classifier scores. A separate packaged diabetes dataset creates regression arrays, a reserved test set, and training-CV MAE comparisons. Hypothesis: nonlinear flexibility may help, but a simpler regularized model may be more stable. MAE is in target units; negative sklearn scoring must be converted back before interpretation.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#3 — CV clones and fits the complete recipe separately inside each training fold.
from sklearn.model_selection import cross_validate
#4 — Logistic regression learns a regularized linear score and class probabilities.
from sklearn.linear_model import LogisticRegression

regularization_rows = []
for strength in [0.05, 1.0, 20.0]:
    #5 — Change only logistic C; smaller C strengthens regularization while max_iter avoids confusing
    #     nonconvergence with underfit.
    candidate = customer_model(LogisticRegression(C=strength, max_iter=1000, random_state=SEED))
    #6 — Fit the entire recipe independently inside each fold; test_score denotes CV exclusions, never the
    #     reserved final X_test.
    scores = cross_validate(candidate, X_train, y_train, cv=cv, scoring='balanced_accuracy')['test_score']
    #7 — Store parameter value, fold mean and spread together so tiny mean changes are not read without
    #     variability.
    regularization_rows.append({'C': strength, 'CV mean': scores.mean(), 'CV SD': scores.std()})
#8 — Convert equally keyed rows or aligned vectors into a named table so comparisons retain their meaning
#     and units.
display(pd.DataFrame(regularization_rows))

<a id="part-08"></a>
# Part 08 — Validation and generalization

[Read the textbook section](https://juandiegopleitez.com/MLstudy/validation.html)

**Learning objective:** Keep preprocessing inside CV and match the split to the intended deployment.

**Step 1.** Compare linear and tree complexity using only the training partition, including preprocessing separately inside every fold. Fold standard deviation describes fold variation, not a confidence interval. **Step 2.** Inspect ordered sensor splits to see how expanding-window validation differs from shuffling future observations into training.

## Guided lesson — The split is part of the scientific question

**Why this matters:** Cross-validation estimates generalization only to observations resembling its excluded folds. Random folds do not answer future prediction or new-person generalization if time or entity identity crosses boundaries. A correct pipeline cannot repair an inappropriate split.

**Expected inputs, outputs, and hypothesis:** Customer training folds yield training/held-out gaps and descriptive fold spread. Ordered sensors yield expanding training windows, later validation positions, and an illustrative two-row embargo. Hypothesis: deeper trees can widen the gap. GroupKFold would be needed for repeated entities even if class proportions became less convenient.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Compare training and excluded-fold scores. A widening gap is diagnostic evidence of overfit, not a guarantee that the shallower model is best.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#3 — CV clones and fits the complete recipe separately inside each training fold.
from sklearn.model_selection import cross_validate
#4 — Logistic regression learns a regularized linear score and class probabilities.
from sklearn.linear_model import LogisticRegression
#5 — Classification trees learn split rules and leaf class proportions.
from sklearn.tree import DecisionTreeClassifier

validation_rows = []
#6 — Compare logistic and two bounded tree depths; keep minimum leaf size fixed to focus the capacity
#     hypothesis.
for name, estimator in [('logistic', LogisticRegression(max_iter=1000, random_state=SEED)),
                         ('tree-2', DecisionTreeClassifier(max_depth=2, min_samples_leaf=5, random_state=SEED)),
                         ('tree-6', DecisionTreeClassifier(max_depth=6, min_samples_leaf=5, random_state=SEED))]:
    #7 — Fit the entire recipe independently inside each fold; test_score denotes CV exclusions, never the
    #     reserved final X_test.
    scores = cross_validate(customer_model(estimator), X_train, y_train, cv=cv,
                            scoring='balanced_accuracy', return_train_score=True)
    #8 — Store train and excluded-fold means side by side; the gap is an overfitting diagnostic, not proof
    #     of a causal explanation.
    validation_rows.append({'model': name, 'train': scores['train_score'].mean(),
                            'held-out folds': scores['test_score'].mean(),
                            'fold SD': scores['test_score'].std()})
#9 — Convert equally keyed rows or aligned vectors into a named table so comparisons retain their meaning
#     and units.
display(pd.DataFrame(validation_rows))

### Part 08 — guided step 2

Inspect chronological positional indexes with an embargo. Explain why the required real-world gap depends on label/feature availability rather than a convenient row count.

In [ ]:
#1 — Forward splits keep validation after training in row order, not random order.
from sklearn.model_selection import TimeSeriesSplit

#2 — n_splits=3 creates expanding windows; gap=2 embargoes boundary rows; returned arrays are integer
#     positions, not row labels.
for train_ids, future_ids in TimeSeriesSplit(n_splits=3, gap=2).split(sensors):
    #3 — A mechanical chronology invariant ensures all training positions precede future positions; still
    #     audit feature availability separately.
    assert train_ids.max() < future_ids.min()
    print(f'Train 0..{train_ids.max()}, gap=2, future {future_ids.min()}..{future_ids.max()}')

### Try it — Part 08

Explain when GroupKFold would be necessary (for example, repeated snapshots of the same customer). Repeat CV with three stratified folds and record how estimates change. Do not select a split because it gives the best score.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 08's evidence boundary. Customer training folds yield training/held-out gaps and descriptive fold spread. Ordered sensors yield expanding training windows, later validation positions, and an illustrative two-row embargo. Hypothesis: deeper trees can widen the gap. GroupKFold would be needed for repeated entities even if class proportions became less convenient.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Cross-validation estimates generalization only to observations resembling its excluded folds. Random folds do not answer future prediction or new-person generalization if time or entity identity crosses boundaries. A correct pipeline cannot repair an inappropriate split.

**Expected output and reading:** Customer training folds yield training/held-out gaps and descriptive fold spread. Ordered sensors yield expanding training windows, later validation positions, and an illustrative two-row embargo. Hypothesis: deeper trees can widen the gap. GroupKFold would be needed for repeated entities even if class proportions became less convenient.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — clone copies configuration while discarding learned state, preventing fitted-state leakage.
from sklearn.base import clone
#2 — Stratified folds approximately preserve class proportions while excluding scored rows.
from sklearn.model_selection import StratifiedKFold
#3 — CV clones and fits the complete recipe separately inside each training fold.
from sklearn.model_selection import cross_validate

#4 — Three stratified folds alter training sample sizes and exclusions under a fixed seed, probing split
#     sensitivity rather than seeking the luckiest score.
cv_three = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
#5 — clone removes learned coefficients and preparation; fold fits learn from their own training subsets
#     only.
print('Three-fold logistic scores:', cross_validate(
    clone(logistic), X_train, y_train, cv=cv_three,
    scoring='balanced_accuracy')['test_score'].round(3))

<a id="part-09"></a>
# Part 09 — Similarity, clustering, and reduction

[Read the textbook section](https://juandiegopleitez.com/MLstudy/similarity-clustering.html)

**Learning objective:** Make representation choices explicit before neighbor search, clustering, and projection.

**Step 1.** Standardize the built-in Iris features for an exploratory unsupervised analysis, deliberately excluding species labels. This is not a held-out supervised score. **Step 2.** Compare k-means, agglomerative clustering, and Gaussian-mixture assignments using silhouette and cluster sizes; none proves that a natural grouping exists. Project with PCA only for visualization.

## Guided lesson — Distance, density, and projection are different assumptions

**Why this matters:** K-means favors compact centroid-based groups; hierarchical clustering merges local groups; Gaussian mixtures model component densities. Scale affects them all. PCA shows high-variance directions, not verified clusters; a two-dimensional picture can hide important original geometry.

**Expected inputs, outputs, and hypothesis:** A (150,4) Iris measurement matrix becomes standardized exploratory geometry, partitions, silhouette scores and a (150,2) PCA view. No species targets are used. Hypothesis: algorithms may disagree because their shape assumptions differ. A separate neighbor classifier uses training-only customer preparation and validation scoring.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Compare three exploratory geometries on standardized Iris features. Expect silhouette and cluster-size reports, not supervised accuracy.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#3 — Counter counts cluster memberships so a score can be interpreted alongside group sizes.
from collections import Counter
#4 — These packaged datasets are installed locally, not fetched during execution.
from sklearn import datasets
#5 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics
#6 — StandardScaler learns feature means/scales; transform reuses rather than updates that state.
from sklearn.preprocessing import StandardScaler
#7 — K-means partitions by squared Euclidean proximity to learned centroids.
from sklearn.cluster import KMeans
#8 — Agglomerative clustering merges groups before cutting a hierarchy.
from sklearn.cluster import AgglomerativeClustering
#9 — A mixture fits weighted Gaussian densities rather than centroid-only assignments.
from sklearn.mixture import GaussianMixture

#10 — load_iris returns a local container; only its (150,4) measurements enter exploratory unsupervised
#     modeling.
iris = datasets.load_iris()
#11 — fit_transform learns scale from all exploratory Iris rows, allowed here only because this is not a
#     supervised holdout evaluation.
Z = StandardScaler().fit_transform(iris.data)
#12 — Three k-means groups and n_init=10 reduce start sensitivity; hierarchy and mixture specify three
#     groups/components with different geometry.
cluster_models = {'k-means': KMeans(n_clusters=3, n_init=10, random_state=SEED),
                  #13 — Agglomeration cuts a merge hierarchy; a mixture uses three Gaussian densities, which
                  #     need not match real species.
                  'hierarchical': AgglomerativeClustering(n_clusters=3),
                  'mixture': GaussianMixture(n_components=3, random_state=SEED)}
cluster_rows = []
for name, model in cluster_models.items():
    #14 — Learn structure and assign the same exploratory rows; labels is an (n,) membership vector, not
    #     held-out predictions.
    labels = model.fit_predict(Z)
    #15 — Silhouette combines within/between distances; Counter exposes group sizes so a favorable score
    #     cannot hide tiny partitions.
    cluster_rows.append({'model': name, 'silhouette': metrics.silhouette_score(Z, labels),
                         'sizes': dict(Counter(labels.tolist()))})
#16 — Convert equally keyed rows or aligned vectors into a named table so comparisons retain their meaning
#     and units.
display(pd.DataFrame(cluster_rows))

### Part 09 — guided step 2

Learn two PCA axes, project, display and close the figure. Read retained variance before interpreting visual separation; the projection is not the original geometry.

In [ ]:
#1 — PCA projects onto learned high-variance orthogonal directions, not verified classes.
from sklearn.decomposition import PCA
#2 — Local plotting renders then closes a figure without saving an artifact.
import matplotlib.pyplot as plt

#3 — n_components=2 learns two directions in standardized four-dimensional space; fit stores axes and
#     centering state.
pca = PCA(n_components=2).fit(Z)
#4 — transform maps (150,4) into (150,2) using fixed axes without refitting to plotted assignments.
projection = pca.transform(Z)
#5 — figsize sets rendering inches, not model units; a local display figure is created without a saved file.
plt.figure(figsize=(5, 3))
#6 — Each coordinate slice has shape (150,); c supplies k-means labels and s is marker area, not a feature
#     weight.
plt.scatter(projection[:, 0], projection[:, 1], c=cluster_models['k-means'].labels_, s=18)
#7 — Label the figure exploratory so visual separation is not mistaken for verified ground truth.
plt.title('Exploratory PCA view; colors are k-means assignments')
#8 — Axis labels describe projection coordinates; tight_layout arranges text, show renders, close releases
#     resources for repeated execution.
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.tight_layout(); plt.show(); plt.close()
#9 — The two ratios measure retained original variance, not predictive accuracy or cluster validity.
print('Explained variance:', pca.explained_variance_ratio_.round(3))

### Try it — Part 09

Change k in the neighbor classifier. Explain the role of scale and why a two-dimensional picture cannot validate the original geometry. Find nearest Iris neighbors excluding the query itself.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 09's evidence boundary. A (150,4) Iris measurement matrix becomes standardized exploratory geometry, partitions, silhouette scores and a (150,2) PCA view. No species targets are used. Hypothesis: algorithms may disagree because their shape assumptions differ. A separate neighbor classifier uses training-only customer preparation and validation scoring.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

Change k while preserving customer training/validation boundaries. Predict the locality-vs-smoothing trade-off before moving the control.

In [ ]:
#1 — Widget constructors specify frontend controls; guided_control runs the same default as ordinary Python first.
import ipywidgets as widgets
#2 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics
#3 — A neighbor classifier votes among stored training examples in transformed feature space.
from sklearn.neighbors import KNeighborsClassifier

#4 — The scalar k changes vote locality; use the original customer training/validation boundary, not
#     globally scaled Iris data.
def neighbor_experiment(k=5):
    #5 — n_neighbors=k stores/votes among scaled training rows; fit does not learn a parametric probability
    #     equation.
    model = customer_model(KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    #6 — predict yields one hard label per row; balanced accuracy averages class-specific recalls instead of
    #     favoring the majority.
    print('Validation balanced accuracy:', round(metrics.balanced_accuracy_score(
        y_val, model.predict(X_val)), 3))
#7 — The helper executes the declared default, then registers callbacks; it returns no replacement final
#     model/policy. IntSlider uses discrete values; odd k steps avoid most binary vote ties.
guided_control(neighbor_experiment, k=widgets.IntSlider(min=1, max=15, step=2, value=5, description='Neighbors'))

### Part 09 — guided step 2

Inspect one Iris query in the exploratory space. Preserve a two-dimensional query shape and exclude the zero-distance self-match from the neighbor report.

In [ ]:
#1 — Neighbor search retrieves by distance without a supervised target.
from sklearn.neighbors import NearestNeighbors

#2 — fit builds an unsupervised search; Z[[0]] keeps query shape (1,4); kneighbors returns distance/index
#     arrays each shape (1,4).
distances, neighbors = NearestNeighbors(n_neighbors=4).fit(Z).kneighbors(Z[[0]])
#3 — Choose query row 0 and skip the first zero-distance self-match; remaining indexes identify original
#     exploratory rows.
print('Nearest non-self rows:', neighbors[0, 1:].tolist(), 'distances:', distances[0, 1:].round(3))

<a id="part-10"></a>
# Part 10 — Evaluation and decisions

[Read the textbook section](https://juandiegopleitez.com/MLstudy/evaluation.html)

**Learning objective:** Choose an action threshold on validation, then report an untouched final test once.

**Step 1.** Use the fixed logistic baseline to estimate validation probabilities. Assign a false negative a cost of 5 and a false positive a cost of 1; minimize this declared illustrative cost on validation only. The widget previews validation trade-offs and does not mutate the frozen policy. **Step 2.** Review precision, recall, confusion counts, ranking, calibration error, and lift before opening test results.

## Guided lesson — A score, a probability, and an action have different evaluations

**Why this matters:** ROC AUC/AP measure ranking, Brier loss measures probability error, and a threshold creates a policy whose costs depend on the decision. Prevalence affects precision and AP. Tiny samples make both calibration bins and cost-optimal cutoffs unstable.

**Expected inputs, outputs, and hypothesis:** Training-fitted logistic probabilities and validation labels select a cutoff from a declared grid. The widget previews validation consequences without mutating the frozen choice. Only afterward do we report final classification and CV-selected regression tests once. Hypothesis: a higher false-negative cost favors more positive actions; never verify this by tuning on test labels.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Compute validation-only expected cost for a declared threshold grid. Freeze the first minimum once; print the selected scalar before any final test calculation.

In [ ]:
#1 — NumPy expresses the numeric arrays, elementwise operations and explicit shape contract in this cell.
import numpy as np
#2 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics

#3 — Frozen training pipeline returns (n_validation,2) probabilities; column 1 is the positive class under
#     classes [0,1].
validation_probability = logistic.predict_proba(X_val)[:, 1]
#4 — Nineteen fixed candidates from .05 to .95 are decision policies, not coefficients or new model fits.
threshold_grid = np.linspace(0.05, 0.95, 19)
#5 — Accept aligned labels/actions and return a scalar total cost; false negatives cost five times false
#     positives in this declared scenario.
def decision_cost(labels, predicted):
    #6 — labels=[0,1] fixes a 2x2 layout; ravel yields TN,FP,FN,TP even when a policy predicts only one
    #     class.
    tn, fp, fn, tp = metrics.confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    #7 — The weight five is an assumed consequence ratio, not an observed monetary estimate or
    #     class-frequency correction.
    return int(fp + 5 * fn)
#8 — Broadcast each candidate threshold against the validation score vector; only validation outcomes
#     determine policy cost.
validation_costs = [decision_cost(y_val, validation_probability >= t) for t in threshold_grid]
#9 — argmin chooses the first minimum on ties; storing an ordinary float isolates the selected policy from
#     widget previews.
frozen_threshold = float(threshold_grid[np.argmin(validation_costs)])
print('Frozen validation-selected threshold:', round(frozen_threshold, 3))

### Part 10 — guided step 2

Preview precision, recall and cost on validation. This callback deliberately does not mutate frozen_threshold or inspect X_test/y_test.

In [ ]:
#1 — Widget constructors specify frontend controls; guided_control runs the same default as ordinary Python first.
import ipywidgets as widgets
#2 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics

#3 — Preview a validation policy but return no replacement threshold and never reference test outcomes.
def threshold_preview(threshold=0.5):
    #4 — An elementwise comparison turns a probability vector into Boolean decisions without changing the
    #     fitted model.
    predicted = validation_probability >= threshold
    #5 — Precision conditions on flagged rows; recall conditions on positives; zero_division=0 explicitly
    #     handles no-positive predictions.
    print({'validation_precision': round(metrics.precision_score(y_val, predicted, zero_division=0), 3),
           'validation_recall': round(metrics.recall_score(y_val, predicted, zero_division=0), 3),
           'validation_cost': decision_cost(y_val, predicted)})
#6 — The helper executes the declared default, then registers callbacks; it returns no replacement final
#     model/policy. FloatSlider min/max/step bound the scenario and description labels it; readout_format
#     affects display only.
guided_control(threshold_preview, threshold=widgets.FloatSlider(
    min=0.05, max=0.95, step=0.05, value=0.5, description='Threshold'))

### Part 10 — guided step 3

Inspect calibration-bin agreement and top-fifth lift. The first checks probability quality, while the second checks ranked concentration; neither estimates treatment effect.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#3 — NumPy expresses the numeric arrays, elementwise operations and explicit shape contract in this cell.
import numpy as np
#4 — Calibration bins compare mean probabilities with observed event fractions.
from sklearn.calibration import calibration_curve

#5 — Quantile bins aim for similar counts; n_bins=4 is tiny-sample diagnostic resolution, not a probability
#     calibrator fit.
observed, mean_probability = calibration_curve(y_val, validation_probability, n_bins=4, strategy='quantile')
#6 — Pair each bin mean probability with its observed fraction; gaps diagnose possible miscalibration but
#     are noisy here.
display(pd.DataFrame({'mean predicted': mean_probability, 'observed fraction': observed}))
#7 — argsort of negative scores ranks descending; [:n//5] selects about the top fifth by integer position
#     with at least one row.
top_ids = np.argsort(-validation_probability)[:max(1, len(y_val) // 5)]
#8 — iloc applies ranked positions to aligned labels; divide selected prevalence by full validation
#     prevalence, not by accuracy.
print('Validation top-fifth lift:', round(y_val.iloc[top_ids].mean() / y_val.mean(), 3))

### Try it — Part 10

Before running this cell, write down the fixed threshold and cost assumptions. This is the single final classification and regression test report. If you change choices after seeing it, treat these results as development evidence and obtain a new holdout; a widget is not permission to optimize against test outcomes. How would delayed labels affect monitoring?

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 10's evidence boundary. Training-fitted logistic probabilities and validation labels select a cutoff from a declared grid. The widget previews validation consequences without mutating the frozen choice. Only afterward do we report final classification and CV-selected regression tests once. Hypothesis: a higher false-negative cost favors more positive actions; never verify this by tuning on test labels.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

Open the classification final holdout only after the policy is frozen. Interpret confusion counts, ranking, probability loss and cost as distinct outputs.

In [ ]:
#1 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics

#2 — Open the untouched customer test once for scored evaluation; no training, refitting or threshold search
#     happens here.
test_probability = logistic.predict_proba(X_test)[:, 1]
#3 — Apply the validation-frozen policy without optimizing on test labels.
test_prediction = test_probability >= frozen_threshold
print('FINAL fixed-policy classification test')
#4 — Specify label order so matrix rows/columns stay comparable with the declared-cost calculation.
print(metrics.confusion_matrix(y_test, test_prediction, labels=[0, 1]))
#5 — Report fixed-policy recalls and precision separately from ranking/calibration scores; zero_division is
#     an explicit edge-case convention.
print({'balanced_accuracy': metrics.balanced_accuracy_score(y_test, test_prediction),
       'precision': metrics.precision_score(y_test, test_prediction, zero_division=0),
       'recall': metrics.recall_score(y_test, test_prediction, zero_division=0),
       #6 — ROC AUC measures positive-vs-negative ordering; average precision summarizes positive ranking
       #     and depends on class prevalence.
       'ROC_AUC': metrics.roc_auc_score(y_test, test_probability),
       'average_precision': metrics.average_precision_score(y_test, test_probability),
       #7 — Mean squared probability error is lower-is-better; thresholded mistake cost answers a different
       #     operational question.
       'Brier_loss': metrics.brier_score_loss(y_test, test_probability),
       'declared_cost': decision_cost(y_test, test_prediction)})

### Part 10 — guided step 2

Clone the training-CV regression winner, fit on all regression training data, and test once. A finite holdout may disagree with the CV ranking; do not retune to erase that disagreement.

In [ ]:
#1 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics
#2 — clone copies configuration while discarding learned state, preventing fitted-state leakage.
from sklearn.base import clone

#3 — Read the first row of the training-CV MAE ranking; no final regression error has influenced selection.
regression_winner = str(regression_results.iloc[0]['model'])
#4 — Clone configuration, fit on all regression training rows, and preserve any pipeline preprocessing
#     during prediction.
final_regressor = clone(regressors[regression_winner]).fit(X_reg_train, y_reg_train)
#5 — One numeric prediction per held-out regression row is produced without learning from its target.
regression_prediction = final_regressor.predict(X_reg_test)
print('FINAL regression test, CV-selected model:', regression_winner)
#6 — MAE is mean absolute residual; RMSE emphasizes larger errors; R2 compares residual variation with a
#     mean reference and may be negative.
print({'MAE': metrics.mean_absolute_error(y_reg_test, regression_prediction),
       'RMSE': metrics.root_mean_squared_error(y_reg_test, regression_prediction),
       'R2': metrics.r2_score(y_reg_test, regression_prediction)})

<a id="part-11"></a>
# Part 11 — Text mining

[Read the textbook section](https://juandiegopleitez.com/MLstudy/text-mining.html)

**Learning objective:** Keep vocabulary learning inside the training split and distinguish retrieval from classification.

**Step 1.** Build a tiny original support-message corpus. TF–IDF rarity is not supervised information gain; the corpus is intentionally too small for generalization claims. **Step 2.** Fit TF–IDF plus multinomial naive Bayes and logistic regression on training messages and compare a held-out validation slice. No pretrained language models or downloads are needed.

## Guided lesson — Vocabulary learning is preprocessing

**Why this matters:** TF-IDF learns vocabulary and inverse-document-frequency weights, so fitting it on held-out text leaks representation information. Unigrams and bigrams encode words and adjacent pairs, not syntax understanding. Rarity is not supervised information gain; retrieval is not classification or entity recognition.

**Expected inputs, outputs, and hypothesis:** Sixteen original message/label pairs become train/validation text arrays, sparse term matrices and two fitted pipelines. A new message yields a known topic and lexical neighbors; SVD yields two latent coordinates. Hypothesis: toy vocabulary overlap makes scores look strong, while unseen phrasing exposes the narrow support of the corpus.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Inspect the original (message,topic) supervision, transpose it into aligned arrays, then split before learning vocabulary. Expect twelve training and four held-out messages.

In [ ]:
#1 — This part will learn TF-IDF only after splitting; importing the recipe does not learn a vocabulary.
from sklearn.feature_extraction.text import TfidfVectorizer
#2 — NumPy expresses the numeric arrays, elementwise operations and explicit shape contract in this cell.
import numpy as np
#3 — Split aligned features/labels before learning representation statistics.
from sklearn.model_selection import train_test_split

#4 — Original (message,topic) tuples preserve supervised alignment; this miniature corpus is not a language
#     benchmark.
text_examples = [
 ('invoice charge looks wrong', 'billing'), ('payment was charged twice', 'billing'),
 ('refund for duplicate payment', 'billing'), ('invoice payment receipt missing', 'billing'),
 ('billing charge needs refund', 'billing'), ('monthly invoice amount changed', 'billing'),
 ('payment receipt for charge', 'billing'), ('refund invoice payment please', 'billing'),
 ('login password reset failed', 'access'), ('account login is blocked', 'access'),
 ('password reset link expired', 'access'), ('cannot access my account', 'access'),
 ('login account password help', 'access'), ('reset account access please', 'access'),
 ('password account locked again', 'access'), ('access login reset link', 'access'),
]
#5 — zip(*) transposes pairs; map(np.array,...) creates aligned one-dimensional text/label arrays for
#     splitting.
texts, text_labels = map(np.array, zip(*text_examples))
#6 — Reserve 25% with class stratification before learning vocabulary or document frequencies.
text_train, text_val, label_train, label_val = train_test_split(
    texts, text_labels, test_size=0.25, stratify=text_labels, random_state=SEED)

### Part 11 — guided step 2

Fit separate TF-IDF+NB and TF-IDF+logistic pipelines. Compare macro F1 on the same tiny validation set, then explain why vocabulary repetition makes this weak real-world evidence.

In [ ]:
#1 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics
#2 — make_pipeline derives lowercase step names and chains fit/transform/predict responsibilities.
from sklearn.pipeline import make_pipeline
#3 — Logistic regression learns a regularized linear score and class probabilities.
from sklearn.linear_model import LogisticRegression
#4 — Multinomial NB consumes nonnegative token evidence under approximate independence.
from sklearn.naive_bayes import MultinomialNB
#5 — TF-IDF learns vocabulary/rarity from training text and returns sparse term features.
from sklearn.feature_extraction.text import TfidfVectorizer

#6 — Store whole fitted pipelines by name so future raw messages use exactly the corresponding learned
#     vocabulary and estimator.
text_models = {}
#7 — Compare count-evidence NB and regularized linear classification on the same text split rather than
#     unrelated corpora.
for name, estimator in [('NB', MultinomialNB()), ('logistic', LogisticRegression(random_state=SEED))]:
    #8 — ngram_range=(1,2) includes tokens and adjacent pairs; fit learns vocabulary/IDF then trains on
    #     sparse nonnegative term features.
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), estimator).fit(text_train, label_train)
    #9 — Keep the fitted text-to-label interface, not only classifier coefficients detached from their
    #     feature ordering.
    text_models[name] = model
    #10 — Macro F1 averages label-specific precision/recall harmonic means equally; four held-out messages
    #     cannot establish robustness.
    print(name, 'toy validation macro F1:', metrics.f1_score(
        label_val, model.predict(text_val), average='macro'))

### Try it — Part 11

Type a new synthetic request. Compare the two classifiers and retrieve similar training documents. Inspect a two-component SVD as a latent representation, not as named entities, verified topics, or semantic understanding. Real entity recognition requires annotated spans and task-specific evaluation.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 11's evidence boundary. Sixteen original message/label pairs become train/validation text arrays, sparse term matrices and two fitted pipelines. A new message yields a known topic and lexical neighbors; SVD yields two latent coordinates. Hypothesis: toy vocabulary overlap makes scores look strong, while unseen phrasing exposes the narrow support of the corpus.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

Classify one synthetic raw message with both already fitted pipelines. An out-of-vocabulary message still receives a known label; that is not proof of understanding.

In [ ]:
#1 — Widget constructors specify frontend controls; guided_control runs the same default as ordinary Python first.
import ipywidgets as widgets

#2 — Accept one string and wrap it as [message], a one-document collection rather than an invalid scalar
#     string input.
def classify_message(message='please refund the invoice payment'):
    #3 — Each predict returns a shape-(1,) topic array; [0] extracts one label for reporting without
    #     training on the new message.
    print({name: model.predict([message])[0] for name, model in text_models.items()})
#4 — The helper executes the declared default, then registers callbacks; it returns no replacement final
#     model/policy. Text accepts a raw synthetic string; its callback predicts rather than relearning
#     vocabulary.
guided_control(classify_message, message=widgets.Text(value='please refund the invoice payment', description='Message'))

### Part 11 — guided step 2

Reuse the NB training vocabulary for lexical retrieval. Transform, never fit, the query; inspect training texts in decreasing cosine similarity.

In [ ]:
#1 — Cosine compares vector direction; normalized TF-IDF supports lexical retrieval.
from sklearn.metrics.pairwise import cosine_similarity

#2 — named_steps uses the automatic lowercase class name to access the NB pipeline training-fitted
#     vocabulary/IDF.
vectorizer = text_models['NB'].named_steps['tfidfvectorizer']
#3 — transform returns sparse (n_training_documents,vocabulary_size) features without adding words or
#     relearning IDF.
document_vectors = vectorizer.transform(text_train)
#4 — A one-item list preserves document collection shape (1,vocabulary_size); unseen terms cannot add
#     columns.
query_vector = vectorizer.transform(['password reset'])
#5 — Output (1,n_documents) is flattened, argsorted ascending, reversed, then truncated to three highest
#     similarities.
ranking = cosine_similarity(query_vector, document_vectors).ravel().argsort()[::-1][:3]
#6 — Position indexes retrieve original training texts in ranked order; similarity is lexical evidence, not
#     a relevance annotation.
print('Retrieved:', text_train[ranking].tolist())

### Part 11 — guided step 3

Compress sparse training term features into two dense SVD coordinates. Shape is a representation check, not a topic-quality or entity-recognition evaluation.

In [ ]:
#1 — SVD compresses sparse term features without centering them into a dense matrix.
from sklearn.decomposition import TruncatedSVD

#2 — Two SVD components compress sparse training features to dense (n_documents,2) coordinates without
#     centering.
latent = TruncatedSVD(n_components=2, random_state=SEED).fit_transform(document_vectors)
#3 — A shape check validates the representation contract, not semantic topic quality.
print('Toy latent representation:', latent.shape)

<a id="part-12"></a>
# Part 12 — Associations and ensembles

[Read the textbook section](https://juandiegopleitez.com/MLstudy/associations-ensembles.html)

**Learning objective:** Calculate association strength and compare ensembles without causal claims.

**Step 1.** Compute support, confidence, and lift from original toy baskets. Joint occurrence is descriptive: lift is not an intervention effect. **Step 2.** Compare bagging-style random forests, gradient boosting, and soft voting against a dummy using the original customer training CV, not the already opened test set. This is a separate family comparison, not a replacement final model.

## Guided lesson — Associations describe transactions; ensembles combine predictors

**Why this matters:** Association support is joint frequency; confidence conditions on an antecedent; lift compares confidence with the consequent base rate. None estimates the causal effect of buying one item. Forests average randomized trees, boosting adds trees sequentially, and soft voting averages class probabilities.

**Expected inputs, outputs, and hypothesis:** Eight original baskets become rule statistics; customer training folds become dummy/forest/boosting/voting scores. Hypothesis: nonlinear ensembles may capture interactions but need not beat a simpler model on small samples. A one-basket change tests denominator sensitivity; final tests already used are not reopened.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Derive support, directional confidence and lift from set containment. Keep all ratios tied to the same transaction denominator.

In [ ]:
#1 — These local ensemble recipes are fitted in the predictive-comparison step after separating descriptive association evidence.
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
#2 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#3 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#4 — combinations enumerates unordered item pairs once; rule direction is stated separately.
from itertools import combinations

#5 — Sets discard duplicate quantities and item order, matching presence/absence transaction semantics.
baskets = [set(x) for x in [('tea', 'bread'), ('tea', 'milk'), ('bread', 'milk'),
                             ('tea', 'bread', 'milk'), ('bread',), ('tea', 'bread'),
                             ('milk',), ('tea', 'milk')]]
#6 — Accept an item iterable and return its empirical fraction among the enclosing original basket
#     collection.
def support(items):
    #7 — Set <= means subset containment; summing Boolean containment results counts supporting transactions.
    return sum(set(items) <= basket for basket in baskets) / len(baskets)
association_rows = []
#8 — Enumerate unordered pairs once; choosing a then b below defines the displayed directional confidence
#     rule.
for a, b in combinations(['tea', 'bread', 'milk'], 2):
    #9 — Compute joint support on the same basket denominator later used for both marginal supports.
    joint = support([a, b])
    #10 — Confidence=joint/P(a); lift=joint/[P(a)P(b)]; all listed denominators are positive, but rare
    #     antecedents make ratios unstable.
    association_rows.append({'rule': f'{a} -> {b}', 'support': joint,
                             'confidence': joint / support([a]),
                             'lift': joint / (support([a]) * support([b]))})
#11 — Convert equally keyed rows or aligned vectors into a named table so comparisons retain their meaning
#     and units.
display(pd.DataFrame(association_rows))

### Part 12 — guided step 2

Compare declared forest, boosting and soft-voting recipes against a dummy with training CV only. Read both mean and spread, not just the largest number.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#3 — CV clones and fits the complete recipe separately inside each training fold.
from sklearn.model_selection import cross_validate
#4 — A feature-ignoring class baseline tests whether informative modeling adds value.
from sklearn.dummy import DummyClassifier
#5 — Logistic regression learns a regularized linear score and class probabilities.
from sklearn.linear_model import LogisticRegression
#6 — Classification trees learn split rules and leaf class proportions.
from sklearn.tree import DecisionTreeClassifier
#7 — Forests average bootstrapped trees with randomized feature choices at splits.
from sklearn.ensemble import RandomForestClassifier
#8 — Histogram boosting improves a loss by adding bounded trees sequentially.
from sklearn.ensemble import HistGradientBoostingClassifier
#9 — Soft voting averages constituent class probabilities, which may still need calibration.
from sklearn.ensemble import VotingClassifier

#10 — Names map to unfitted models; the prior-only dummy anchors claims that more complexity adds predictive
#     value.
ensemble_estimators = {
    'dummy': DummyClassifier(strategy='prior'),
    #11 — Sixty bootstrapped randomized trees are depth-limited to four; fixed seed stabilizes sampling and
    #     n_jobs=1 controls resource use.
    'forest': RandomForestClassifier(n_estimators=60, max_depth=4, random_state=SEED, n_jobs=1),
    #12 — Sixty sequential stages with at most seven leaf nodes per tree bound boosting complexity and
    #     execution cost.
    'boosting': HistGradientBoostingClassifier(max_iter=60, max_leaf_nodes=7, random_state=SEED),
    #13 — Named estimator tuples define independent linear/tree recipes; soft voting averages probabilities,
    #     not hard labels.
    'soft vote': VotingClassifier(estimators=[
        ('linear', LogisticRegression(max_iter=1000, random_state=SEED)),
        ('tree', DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=SEED))], voting='soft'),
}
ensemble_rows = []
for name, estimator in ensemble_estimators.items():
    #14 — Wrap each ensemble in fresh preparation and evaluate only training folds; previously opened final
    #     test remains closed.
    score = cross_validate(customer_model(estimator), X_train, y_train, cv=cv,
                           scoring='balanced_accuracy')['test_score']
    #15 — Fold mean and SD summarize both typical score and split sensitivity; neither alone establishes a
    #     stable winner.
    ensemble_rows.append({'model': name, 'CV balanced accuracy': score.mean(), 'fold SD': score.std()})
#16 — Convert equally keyed rows or aligned vectors into a named table so comparisons retain their meaning
#     and units.
display(pd.DataFrame(ensemble_rows))

### Try it — Part 12

Change one basket, recompute one rule, and explain the effect of a tiny denominator. Name a confounder that could explain tea–bread co-occurrence. Larger ensembles are not automatically better; compare their evidence and operational costs.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 12's evidence boundary. Eight original baskets become rule statistics; customer training folds become dummy/forest/boosting/voting scores. Hypothesis: nonlinear ensembles may capture interactions but need not beat a simpler model on small samples. A one-basket change tests denominator sensitivity; final tests already used are not reopened.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Association support is joint frequency; confidence conditions on an antecedent; lift compares confidence with the consequent base rate. None estimates the causal effect of buying one item. Forests average randomized trees, boosting adds trees sequentially, and soft voting averages class probabilities.

**Expected output and reading:** Eight original baskets become rule statistics; customer training folds become dummy/forest/boosting/voting scores. Hypothesis: nonlinear ensembles may capture interactions but need not beat a simpler model on small samples. A one-basket change tests denominator sensitivity; final tests already used are not reopened.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Concatenation creates a new collection, leaving support() tied to the unchanged original reference.
changed_baskets = baskets + [{'tea', 'bread'}]
#2 — Count joint containment under the changed collection and its own denominator; do not mix old/new sample
#     sizes.
changed_joint = sum({'tea', 'bread'} <= b for b in changed_baskets) / len(changed_baskets)
#3 — Antecedent presence defines the confidence conditioning population in the changed sample.
changed_tea = sum('tea' in b for b in changed_baskets) / len(changed_baskets)
print('Confidence tea -> bread before:', support(['tea', 'bread']) / support(['tea']))
#4 — Divide changed joint by changed antecedent support for a coherent sensitivity comparison.
print('Confidence after one added basket:', changed_joint / changed_tea)

<a id="part-13"></a>
# Part 13 — Neural models and anomaly detection

[Read the textbook section](https://juandiegopleitez.com/MLstudy/neural-anomaly.html)

**Learning objective:** Compare neural baselines and isolate anomaly threshold calibration from test data.

**Step 1.** Compare a small scikit-learn MLPClassifier with the logistic baseline on validation. This is **not TensorFlow**, deep-learning certification, or an unrestricted architecture search. **Step 2.** Train IsolationForest and a small reconstruction MLPRegressor only on known-normal early synthetic sensor rows. The latter acts as a toy autoencoder by learning input-to-input reconstruction, not as a dedicated deep autoencoder framework. Fit scale on those same training rows; calibrate reconstruction and forest thresholds on later normal validation rows only.

## Guided lesson — Neural models do not escape data-boundary rules

**Why this matters:** The classifier is a small scikit-learn MLP, not TensorFlow. The reconstruction MLP learns input-to-input mapping through a bottleneck, acting as a toy autoencoder. Known-normal training and calibration data are unusually convenient assumptions; real contamination makes anomaly detection harder.

**Expected inputs, outputs, and hypothesis:** Customer training rows yield a validation comparison. Sensor positions 0–99 supply curated normal training rows, 100–139 normal calibration rows, and later rows a final anomaly test. Hypothesis: both isolation and reconstruction detect large injected shifts. Align score direction and calibrate thresholds only on the separate normal window before inspecting test labels.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Compare a small sklearn neural classifier against logistic on identical validation rows. State the one-layer architecture and do not treat this as final test superiority.

In [ ]:
#1 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics
#2 — This small feed-forward classifier is scikit-learn, not TensorFlow.
from sklearn.neural_network import MLPClassifier

#3 — hidden_layer_sizes=(12,) is one twelve-unit layer; lbfgs optimizes weights; alpha=1 regularizes and
#     max_iter bounds optimization.
mlp = customer_model(MLPClassifier(hidden_layer_sizes=(12,), solver='lbfgs',
                                  alpha=1.0, max_iter=2000, random_state=SEED)).fit(X_train, y_train)
#4 — Compare MLP and logistic on the same held-out rows, not the already used final test; improvement does
#     not justify post-test replacement.
print('Validation balanced accuracy:', {
    'logistic': metrics.balanced_accuracy_score(y_val, logistic.predict(X_val)),
    'MLP': metrics.balanced_accuracy_score(y_val, mlp.predict(X_val))})

### Part 13 — guided step 2

Split ordered sensors into normal training, later normal calibration and final test windows. Fit scaling only on curated early normals.

In [ ]:
#1 — StandardScaler learns feature means/scales; transform reuses rather than updates that state.
from sklearn.preprocessing import StandardScaler

#2 — Only the three measured signals enter detectors; order and is_anomaly remain metadata, not predictive
#     shortcuts.
sensor_features = ['temperature', 'vibration', 'pressure']
#3 — Take early positions then filter known normals; this label-informed curation is a strong synthetic-data
#     advantage.
sensor_train = sensors.iloc[:100].query('is_anomaly == 0')
#4 — A later normal-only window calibrates cutoffs separately from fit; it cannot inform training scale.
sensor_validation = sensors.iloc[100:140].query('is_anomaly == 0')
#5 — Keep all later rows, including outliers; filtering evaluation to normal rows would destroy anomaly
#     recall assessment.
sensor_test = sensors.iloc[140:]
#6 — Fit means/scales on curated normal training rows only, not on shifted calibration or test values.
sensor_scaler = StandardScaler().fit(sensor_train[sensor_features])
#7 — transform produces (n_normal_training_rows,3) numeric features using stored normal-only means/scales.
S_train = sensor_scaler.transform(sensor_train[sensor_features])
#8 — Calibration transformation applies training state without updating it, preserving a genuine held-out
#     normal window.
S_val = sensor_scaler.transform(sensor_validation[sensor_features])
#9 — Transform test features with the same state; score values must not influence model or threshold choices.
S_test = sensor_scaler.transform(sensor_test[sensor_features])

### Part 13 — guided step 3

Fit isolation and reconstruction scores, align their directions, and freeze a normal-calibration quantile per method. Large scores should mean anomalous for both.

In [ ]:
#1 — NumPy expresses the numeric arrays, elementwise operations and explicit shape contract in this cell.
import numpy as np
#2 — Isolation scores arise from random partitions, not a supplied anomaly target.
from sklearn.ensemble import IsolationForest
#3 — Multiple numeric outputs allow an input-to-input reconstruction baseline.
from sklearn.neural_network import MLPRegressor

#4 — Fit sixty random isolation trees on features alone; no anomaly labels are supplied to the estimator.
forest_anomaly = IsolationForest(n_estimators=60, random_state=SEED).fit(S_train)
#5 — A two-unit tanh bottleneck reconstructs three outputs; fit(S_train,S_train) makes inputs the targets;
#     alpha regularizes the mapping.
autoencoder = MLPRegressor(hidden_layer_sizes=(2,), activation='tanh', solver='lbfgs',
                           alpha=0.1, max_iter=2000, random_state=SEED).fit(S_train, S_train)
#6 — Accept a numeric (n,3) matrix already in fitted scaled units; return one nonnegative scalar score per
#     row.
def reconstruction_error(values):
    #7 — Subtract predicted (n,3) reconstruction, square elementwise, then mean(axis=1) over features rather
    #     than across people/readings.
    return np.mean((values - autoencoder.predict(values)) ** 2, axis=1)
#8 — Negate forest normality so larger is more anomalous for both methods; only held-out normal calibration
#     rows enter this mapping.
anomaly_validation_scores = {'isolation forest': -forest_anomaly.score_samples(S_val),
                              'reconstruction': reconstruction_error(S_val)}
#9 — The 95th percentile freezes an upper-tail cutoff per method; tiny calibration quantiles are unstable
#     and not guaranteed future false-alarm rates.
anomaly_thresholds = {name: float(np.quantile(values, 0.95))
                       for name, values in anomaly_validation_scores.items()}
print('Validation-only 95th-percentile thresholds:', anomaly_thresholds)

### Try it — Part 13

Freeze both thresholds before this one anomaly test report. Compare held-out average precision and recall, and explain why synthetic injected outliers are much easier than real rare failures. Change the calibration quantile only in a new development cycle, not to improve this test.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 13's evidence boundary. Customer training rows yield a validation comparison. Sensor positions 0–99 supply curated normal training rows, 100–139 normal calibration rows, and later rows a final anomaly test. Hypothesis: both isolation and reconstruction detect large injected shifts. Align score direction and calibrate thresholds only on the separate normal window before inspecting test labels.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** The classifier is a small scikit-learn MLP, not TensorFlow. The reconstruction MLP learns input-to-input mapping through a bottleneck, acting as a toy autoencoder. Known-normal training and calibration data are unusually convenient assumptions; real contamination makes anomaly detection harder.

**Expected output and reading:** Customer training rows yield a validation comparison. Sensor positions 0–99 supply curated normal training rows, 100–139 normal calibration rows, and later rows a final anomaly test. Hypothesis: both isolation and reconstruction detect large injected shifts. Align score direction and calibrate thresholds only on the separate normal window before inspecting test labels.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Metric functions compare aligned true outcomes with labels, scores or numeric predictions.
from sklearn import metrics

#2 — Score later held-out rows once with trained detectors; do not change thresholds based on these
#     evaluation outcomes.
anomaly_test_scores = {'isolation forest': -forest_anomaly.score_samples(S_test),
                       'reconstruction': reconstruction_error(S_test)}
for name, scores in anomaly_test_scores.items():
    #3 — Compare scores with the corresponding method-specific calibration cutoff; this changes ranking into
    #     review actions.
    alerts = scores > anomaly_thresholds[name]
    #4 — AP uses score order, while precision/recall use alerts; labels here are only held-out synthetic
    #     evaluation metadata.
    print(name, {'AP': metrics.average_precision_score(sensor_test.is_anomaly, scores),
                 'precision': metrics.precision_score(sensor_test.is_anomaly, alerts, zero_division=0),
                 'recall': metrics.recall_score(sensor_test.is_anomaly, alerts, zero_division=0),
                 #5 — Sum Boolean flags to report review volume alongside quality metrics; workload matters
                 #     even when ranking looks excellent.
                 'alerts': int(alerts.sum())})
print('Normal calibration rows:', len(sensor_validation), '; tiny-sample quantiles are unstable.')

<a id="part-14"></a>
# Part 14 — Optimization

[Read the textbook section](https://juandiegopleitez.com/MLstudy/optimization.html)

**Learning objective:** Check objective, constraints, residuals, and solver status rather than trusting a returned number.

**Step 1.** Solve a two-product linear program: maximize profit 3x + 2y subject to x + y ≤ 4 and 2x + y ≤ 5, with nonnegative quantities. Negate profit for scipy’s minimizer. **Step 2.** Verify feasibility and the known optimum (x=1, y=3, profit=9); then solve a differentiable quadratic with an explicit gradient.

## Guided lesson — Write the mathematics before calling a solver

**Why this matters:** Variables x,y are continuous nonnegative product units. Rows of A encode resource consumption, b gives matching capacities, and a profit vector gives value per unit. scipy minimizes, so only the objective sign is negated. Integer production would need a different formulation.

**Expected inputs, outputs, and hypothesis:** A two-variable LP produces quantities, profit, status and resource slack; a smooth quadratic produces an optimum and gradient norm. Hypothesis: increasing a binding resource helps until another constraint binds. Check feasibility, status and the known solution; a returned vector alone is not evidence the intended problem was solved.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Form and verify the continuous LP. Map each constraint row to an English resource statement, then check success, feasibility and the known optimum.

In [ ]:
#1 — NumPy expresses the numeric arrays, elementwise operations and explicit shape contract in this cell.
import numpy as np
#2 — Linear programming minimizes a linear objective subject to explicit bounds/inequalities.
from scipy.optimize import linprog

#3 — A shape-(2,2) matrix has one resource inequality per row and coefficients ordered [x,y] in both rows.
A = np.array([[1., 1.], [2., 1.]])
#4 — The length-two vector aligns capacities with A rows, not with product variables.
b = np.array([4., 5.])
#5 — Two objective coefficients give value per unit; dot product with quantities computes total positive
#     profit.
profit = np.array([3., 2.])
#6 — Negate only profit because scipy minimizes; A_ub/b_ub impose <= constraints, bounds enforce
#     nonnegativity, highs selects the LP backend.
production = linprog(-profit, A_ub=A, b_ub=b, bounds=[(0, None), (0, None)], method='highs')
#7 — Check solver status before using .x; infeasible, unbounded or numerical failures must not be presented
#     as solutions.
assert production.success, production.message
#8 — Matrix multiplication computes resource usage; compare each inequality with 1e-8 tolerance for
#     floating-point arithmetic.
assert np.all(A @ production.x <= b + 1e-8)
#9 — Check the analytically known optimum [1,3] with numerical tolerance, a stronger test than checking
#     solver success alone.
assert np.allclose(production.x, [1, 3])
#10 — Resource slack b-A@x shows unused capacity; zero slack indicates a binding inequality at the reported
#     solution.
print({'quantities': production.x, 'profit': profit @ production.x,
       'resource_slack': b - A @ production.x, 'status': production.message})

### Part 14 — guided step 2

Solve a separate smooth quadratic with an explicit gradient. The stationary-point check complements, rather than replaces, the objective formulation.

In [ ]:
#1 — NumPy expresses the numeric arrays, elementwise operations and explicit shape contract in this cell.
import numpy as np
#2 — Smooth optimization uses an objective and an optional explicit gradient.
from scipy.optimize import minimize

#3 — A length-two zero guess starts BFGS; objective sums squared distances to [2,-1] and jac provides the
#     matching analytic gradient.
quadratic = minimize(lambda z: np.sum((z - np.array([2., -1.])) ** 2),
                     x0=np.zeros(2), jac=lambda z: 2 * (z - np.array([2., -1.])), method='BFGS')
#4 — Inspect convergence status for smooth optimization separately from the unrelated LP solver result.
assert quadratic.success, quadratic.message
#5 — Gradient norm measures first-order stationarity; a small value complements status but cannot validate a
#     wrong objective formulation.
print('Quadratic optimum:', quadratic.x, 'gradient norm:', np.linalg.norm(quadratic.jac))

### Try it — Part 14

Change the second resource bound using the slider. Verify feasibility again and note where the optimal decision changes. Continuous units are an assumption: if products must be whole units, formulate an integer optimization problem instead.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 14's evidence boundary. A two-variable LP produces quantities, profit, status and resource slack; a smooth quadratic produces an optimum and gradient norm. Hypothesis: increasing a binding resource helps until another constraint binds. Check feasibility, status and the known solution; a returned vector alone is not evidence the intended problem was solved.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Variables x,y are continuous nonnegative product units. Rows of A encode resource consumption, b gives matching capacities, and a profit vector gives value per unit. scipy minimizes, so only the objective sign is negated. Integer production would need a different formulation.

**Expected output and reading:** A two-variable LP produces quantities, profit, status and resource slack; a smooth quadratic produces an optimum and gradient norm. Hypothesis: increasing a binding resource helps until another constraint binds. Check feasibility, status and the known solution; a returned vector alone is not evidence the intended problem was solved.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Widget constructors specify frontend controls; guided_control runs the same default as ordinary Python first.
import ipywidgets as widgets
#2 — NumPy expresses the numeric arrays, elementwise operations and explicit shape contract in this cell.
import numpy as np
#3 — Linear programming minimizes a linear objective subject to explicit bounds/inequalities.
from scipy.optimize import linprog

def capacity_experiment(capacity=5.0):
    #4 — Hold the first resource fixed at four and vary only the second, preserving row ordering in the
    #     constraint matrix.
    limits = np.array([4., capacity])
    #5 — Solve an independent capacity scenario using the same positive-profit interpretation and
    #     nonnegative variable bounds.
    result = linprog(-profit, A_ub=A, b_ub=limits, bounds=[(0, None), (0, None)], method='highs')
    if result.success:
        #6 — Recheck feasibility against changed limits, not original capacities; success alone should not
        #     replace inspecting constraints.
        assert np.all(A @ result.x <= limits + 1e-8)
        #7 — Dot product reports original positive profit; round affects display only and does not alter
        #     solver quantities.
        print('Quantities:', result.x.round(3), 'profit:', round(float(profit @ result.x), 3))
    else:
        print('No verified solution:', result.message)
#8 — The helper executes the declared default, then registers callbacks; it returns no replacement final
#     model/policy. FloatSlider min/max/step bound the scenario and description labels it; readout_format
#     affects display only.
guided_control(capacity_experiment, capacity=widgets.FloatSlider(
    min=2, max=8, step=0.5, value=5, description='Capacity'))

<a id="part-15"></a>
# Part 15 — Synthetic PII and PHI handling

[Read the textbook section](https://juandiegopleitez.com/MLstudy/pii-phi.html)

**Learning objective:** Practice span offsets, minimization, and evaluation with synthetic placeholders only.

**Step 1.** Define a narrow annotation scheme for literal synthetic EMAIL and PATIENT_ID markers. No real identifiers are collected or displayed. **Step 2.** Match spans, replace from right to left to preserve original offsets, and score exact spans against explicit synthetic annotations. Pattern matching is not general named-entity recognition; a missed name or contextual PHI is still a privacy risk.

## Guided lesson — Privacy scope must be narrower than marketing claims

**Why this matters:** Only literal synthetic EMAIL/PATIENT markers are recognized. A span includes start, exclusive end, and entity label; detecting the right type at the wrong boundary is an error. Python offsets index Unicode characters rather than bytes. Pattern matching is not general NER or compliant de-identification.

**Expected inputs, outputs, and hypothesis:** A fabricated note yields labeled spans, right-to-left replacements, and exact-span precision/recall. A NAME marker deliberately remains undetected to demonstrate the limit. Hypothesis: perfect results on two known markers say almost nothing about unseen real PHI. Define independent templates, annotation controls and privacy review before extending the task.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Recognize only the declared placeholder language, record half-open character spans, and replace right-to-left to preserve original coordinates.

In [ ]:
#1 — Regex recognizes the declared literal marker language, not arbitrary personal entities.
import re

#2 — Only literal fabricated markers occur here; no real contact or health information is collected.
synthetic_note = 'Contact <EMAIL_001>; record <PATIENT_001>; status routine.'
#3 — Raw regex preserves \d; (?:EMAIL|PATIENT) groups alternatives without capturing; {3} requires exactly
#     three digits between marker delimiters.
span_pattern = re.compile(r'<(?:EMAIL|PATIENT)_\d{3}>')
#4 — Accept one Python string; return a list of (inclusive_start,exclusive_end,label) tuples in original
#     character coordinates.
def find_placeholder_spans(text):
    #5 — finditer yields ordered match objects; start/end use half-open offsets and group() supplies the
    #     matched synthetic marker.
    return [(m.start(), m.end(), 'EMAIL' if m.group().startswith('<EMAIL') else 'PATIENT_ID')
            for m in span_pattern.finditer(text)]
#6 — Return a new string; immutable strings are rebuilt without modifying a caller buffer in place.
def redact_placeholders(text):
    #7 — Process original spans right-to-left so changed replacement lengths do not invalidate offsets for
    #     remaining earlier spans.
    for start, end, label in reversed(find_placeholder_spans(text)):
        #8 — Keep prefix/suffix and replace only [start:end]; labels stand in for values to illustrate
        #     minimization.
        text = text[:start] + f'[{label}]' + text[end:]
    return text

### Part 15 — guided step 2

Compare complete labeled spans with explicit synthetic gold offsets. A correct entity type with a wrong boundary is not an exact match.

In [ ]:
#1 — A set of manually annotated (start,end,label) tuples defines exact-span truth only for this fixed
#     fabricated note.
gold_spans = {(8, 19, 'EMAIL'), (28, 41, 'PATIENT_ID')}
#2 — Set conversion makes whole span tuples comparable and prevents duplicate detections inflating true
#     positives.
predicted_spans = set(find_placeholder_spans(synthetic_note))
#3 — Set intersection requires correct start, end and entity type; partial overlap does not count.
true_positive = len(gold_spans & predicted_spans)
#4 — Divide exact matches by predicted spans; max(1,...) declares zero for empty predictions rather than
#     raising division-by-zero.
span_precision = true_positive / max(1, len(predicted_spans))
#5 — Divide exact matches by gold spans; recall is especially important when missed sensitive spans have
#     serious consequences.
span_recall = true_positive / max(1, len(gold_spans))
#6 — Exact equality is a positive smoke test on two known markers, not evidence of broad privacy coverage.
assert predicted_spans == gold_spans
print('Redacted synthetic text:', redact_placeholders(synthetic_note))
print({'exact_span_precision': span_precision, 'exact_span_recall': span_recall})

### Try it — Part 15

Add an unsupported synthetic NAME marker and document the missed span. Explain why perfect scores on two known placeholders do not establish privacy, de-identification compliance, or safe handling of clinical narratives. Define restricted annotation access, review, retention, and patient/document-group splitting before a real project.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 15's evidence boundary. A fabricated note yields labeled spans, right-to-left replacements, and exact-span precision/recall. A NAME marker deliberately remains undetected to demonstrate the limit. Hypothesis: perfect results on two known markers say almost nothing about unseen real PHI. Define independent templates, annotation controls and privacy review before extending the task.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Only literal synthetic EMAIL/PATIENT markers are recognized. A span includes start, exclusive end, and entity label; detecting the right type at the wrong boundary is an error. Python offsets index Unicode characters rather than bytes. Pattern matching is not general NER or compliant de-identification.

**Expected output and reading:** A fabricated note yields labeled spans, right-to-left replacements, and exact-span precision/recall. A NAME marker deliberately remains undetected to demonstrate the limit. Hypothesis: perfect results on two known markers say almost nothing about unseen real PHI. Define independent templates, annotation controls and privacy review before extending the task.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
unsupported_note = 'Review <NAME_001> with <EMAIL_001>.'
#1 — Apply the same unmodified recognizer to a new synthetic template without adding a rule after observing
#     its outcome.
remaining = redact_placeholders(unsupported_note)
#2 — This deliberate negative-capability test documents an unsupported marker still present after redaction.
assert '<NAME_001>' in remaining
print('Known limitation: unsupported synthetic NAME marker remains.')
#3 — Policy fields declare synthetic-only scope and required review; they are not actual approvals or
#     compliance certification.
privacy_plan = {'purpose': 'placeholder span mechanics only',
                'collect_real_data': False, 'retain_raw_notes': False,
                'required_real_work_review': 'privacy, legal, clinical and annotation owners'}
print(privacy_plan)

<a id="part-16"></a>
# Part 16 — Python engineering

[Read the textbook section](https://juandiegopleitez.com/MLstudy/python-engineering.html)

**Learning objective:** Separate input validation, prediction, tests, and trusted local persistence.

**Step 1.** Implement a small inference contract that checks required columns, types, finite numeric values, and supported categories while allowing missing values for the trained imputer. **Step 2.** Test both a valid request and invalid input. This contract is illustrative: range limits, authentication, and institutional policy remain additional work.

## Guided lesson — An estimator is not a complete input contract

**Why this matters:** A local interface should reject malformed requests intentionally, rather than depend on opaque estimator failures. Missing values can remain valid because the fitted imputer handles them; negative/infinite numeric values and unsupported categories violate the illustrative operational contract. This can be stricter than the training encoder.

**Expected inputs, outputs, and hypothesis:** A nonempty named feature frame becomes reordered validated inputs and aligned probability/review outputs. Positive and negative smoke tests check behavior. A trusted bundle round-trip occurs only inside TemporaryDirectory. Untrusted pickle/joblib loading can execute code, so persistence is not safe merely because a file has a model extension.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

Define the raw input contract, including allowed missing values and rejected nonfinite/negative/category values. The function returns a reordered copy, not an in-place mutation.

In [ ]:
#1 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import numpy as np
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#3 — Accept a nonempty pandas frame with exactly feature_columns; return a validated reordered copy or raise
#     a useful exception.
def validate_customer_frame(frame):
    #4 — Reject wrong container types and zero rows before obscure downstream shape or attribute errors
    #     occur.
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        raise ValueError('Provide a non-empty DataFrame')
    #5 — Reject missing/extra names; this illustrative check does not enforce duplicate-name uniqueness or a
    #     complete production schema.
    if set(frame.columns) != set(feature_columns):
        raise ValueError('Columns must match the prediction contract')
    #6 — Canonical column order and a copy prevent caller mutation and ambiguity about feature positions.
    result = frame[feature_columns].copy()
    for name in numeric_features:
        #7 — errors=raise rejects malformed numeric strings instead of silently converting bad data into
        #     imputer-compatible missingness.
        result[name] = pd.to_numeric(result[name], errors='raise')
        #8 — Missing values remain allowed; to_numpy(dtype=float) gives finite/range checks a consistent
        #     numeric representation.
        observed = result[name].dropna().to_numpy(dtype=float)
        #9 — Reject infinity or negative observed values under the declared nonnegative numeric contract.
        if not np.isfinite(observed).all() or (observed < 0).any():
            raise ValueError(f'{name} must be finite, nonnegative, or missing')
    #10 — isin validates known categories; missing categories remain allowed for the trained imputer, unlike
    #     unsupported nonmissing strings.
    if not result.channel.dropna().isin(['web', 'store', 'phone']).all():
        raise ValueError('Unsupported channel; review contract before accepting it')
    return result

### Part 16 — guided step 2

Define predict_local(frame, model, threshold). Return an indexed two-column table so every probability and review flag maps to its input row.

In [ ]:
#1 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#2 — Accept raw rows, a trained full pipeline and a scalar policy cutoff; return one probability/action pair
#     per input row.
def predict_local(frame, model, threshold):
    #3 — Fail at the input boundary before any model computation; valid rows are reordered without changing
    #     caller data.
    checked = validate_customer_frame(frame)
    #4 — Binary probability matrix has shape (n,2); [:,1] is positive probability under the known [0,1]
    #     class order.
    probability = model.predict_proba(checked)[:, 1]
    #5 — Aligned vectors form output columns; index=checked.index preserves request-to-response traceability
    #     without adding IDs to features.
    return pd.DataFrame({'probability': probability, 'review': probability >= threshold}, index=checked.index)

### Part 16 — guided step 3

Exercise one valid request and three intentionally malformed ones. Expected exceptions count as rejected inputs; unexpected exceptions remain visible and accidental acceptance fails the test.

In [ ]:
#1 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import numpy as np

#2 — Select/copy three rows for a local interface probe; this does not alter validation data or train a new
#     model.
request = X_val.iloc[:3].copy()
#3 — Use the frozen model/policy through the same validated path intended for future local batch calls.
response = predict_local(request, logistic, frozen_threshold)
#4 — Check output row count and every probability in [0,1]; successful execution alone is not a sufficient
#     behavior test.
assert len(response) == 3 and response.probability.between(0, 1).all()
#5 — drop creates a missing-field case; assign creates unsupported-category and infinite-number cases
#     without altering the good request.
for invalid in [request.drop(columns='channel'), request.assign(channel='unsupported'),
                request.assign(monthly_spend=np.inf)]:
    try:
        validate_customer_frame(invalid)
    #6 — Only expected contract failures are swallowed; acceptance of bad input reaches an explicit failing
    #     assertion.
    except (ValueError, TypeError):
        continue
    raise AssertionError('Invalid request was accepted')
print('Contract and prediction smoke tests passed.')

### Try it — Part 16

Round-trip only this freshly created trusted model bundle in an automatically cleaned temporary directory. Never load untrusted pickle/joblib files: deserialization can execute code. List what would need to be versioned before handing the model to a teammate.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 16's evidence boundary. A nonempty named feature frame becomes reordered validated inputs and aligned probability/review outputs. Positive and negative smoke tests check behavior. A trusted bundle round-trip occurs only inside TemporaryDirectory. Untrusted pickle/joblib loading can execute code, so persistence is not safe merely because a file has a model extension.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** A local interface should reject malformed requests intentionally, rather than depend on opaque estimator failures. Missing values can remain valid because the fitted imputer handles them; negative/infinite numeric values and unsupported categories violate the illustrative operational contract. This can be stricter than the training encoder.

**Expected output and reading:** A nonempty named feature frame becomes reordered validated inputs and aligned probability/review outputs. Positive and negative smoke tests check behavior. A trusted bundle round-trip occurs only inside TemporaryDirectory. Untrusted pickle/joblib loading can execute code, so persistence is not safe merely because a file has a model extension.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — pandas preserves named fields and row alignment for the local data/contract/evidence operation.
import pandas as pd
#2 — Path composes local locations without creating a file or accessing a network.
from pathlib import Path
#3 — Only deserialize artifacts you fully trust; joblib can execute code while loading.
import joblib
#4 — TemporaryDirectory limits model writes to an automatically cleaned context.
import tempfile
#5 — Record the installed implementation version with the temporary model bundle.
import sklearn

#6 — The context manager scopes every model write to an isolated directory and cleans it automatically,
#     including exception paths.
with tempfile.TemporaryDirectory(prefix='mlstudy-local-') as directory:
    #7 — The filename is inside the temporary context; no persistent model is written to the repository or
    #     user home.
    bundle_path = Path(directory) / 'trusted-demo.joblib'
    #8 — Persist trained pipeline plus frozen threshold, schema and implementation version, not detached
    #     estimator coefficients alone.
    joblib.dump({'pipeline': logistic, 'threshold': frozen_threshold,
                 'feature_columns': feature_columns, 'sklearn_version': sklearn.__version__}, bundle_path)
    #9 — Load only the artifact just created here; untrusted serialized Python can execute code during
    #     loading.
    restored = joblib.load(bundle_path)
    #10 — Exercise the restored bundle through the original interface, including validation and action policy.
    restored_response = predict_local(request, restored['pipeline'], restored['threshold'])
    #11 — Compare values, row/column order and dtypes to establish the round-trip did not change observable
    #     inference.
    pd.testing.assert_frame_equal(response, restored_response)
#12 — After context exit, verify cleanup instead of assuming no temporary artifacts remain.
assert not Path(directory).exists()
print('Trusted round-trip passed; temporary files cleaned.')

<a id="part-17"></a>
# Part 17 — Local delivery and monitoring

[Read the textbook section](https://juandiegopleitez.com/MLstudy/delivery.html)

**Learning objective:** Simulate batch delivery, operational monitoring, and incident review without deployment.

**Step 1.** Call the local prediction function as a batch interface; no server, container, cluster, or network endpoint is started. A container image is packaging, CI is a verification boundary, and Kubernetes orchestration is not implemented here. **Step 2.** Compare a synthetic incoming batch with training statistics. Monitor missingness, standardized mean shift, and action rate; these are investigation signals, not proof of concept drift.

## Guided lesson — Monitoring signals trigger investigation, not automatic conclusions

**Why this matters:** This is a local batch function, not a server, container build or Kubernetes deployment. Feature drift may indicate collection issues or population changes, but cannot alone establish concept drift or lower outcome quality. Labels can arrive late, so operational and predictive monitoring have different clocks.

**Expected inputs, outputs, and hypothesis:** A copied validation batch is shifted without changing the original reference. Missing-rate changes, training-standardized mean shifts and review fraction expose different signals. Hypothesis: added missing tenure and raised spend trigger different monitors. Declared investigation thresholds are heuristics; ownership, rollback and delayed-label evidence remain required.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** This is a local batch function, not a server, container build or Kubernetes deployment. Feature drift may indicate collection issues or population changes, but cannot alone establish concept drift or lower outcome quality. Labels can arrive late, so operational and predictive monitoring have different clocks.

**Expected output and reading:** A copied validation batch is shifted without changing the original reference. Missing-rate changes, training-standardized mean shifts and review fraction expose different signals. Hypothesis: added missing tenure and raised spend trigger different monitors. Declared investigation thresholds are heuristics; ownership, rollback and delayed-label evidence remain required.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import numpy as np
#3 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#4 — A new synthetic incoming batch leaves validation reference data unchanged for reproducible comparisons.
incoming = X_val.copy()
#5 — Vector addition raises every spend by 25 units, creating an intentional distribution change without
#     changing column types.
incoming['monthly_spend'] = incoming.monthly_spend + 25
#6 — Set selected indexed rows to missing tenure so missing-rate monitoring can be distinguished from a mean
#     shift.
incoming.loc[incoming.index[:8], 'tenure_months'] = np.nan
#7 — Score the local batch using validated input and the frozen policy; no endpoint, container or real
#     deployment is started.
batch_response = predict_local(incoming, logistic, frozen_threshold)
#8 — Columnwise Series align by feature names, making one monitoring row per field rather than per person.
monitoring = pd.DataFrame({
    #9 — isna().mean() turns Boolean masks into fractions; compare incoming values with a fixed training
    #     reference.
    'training_missing_rate': X_train[numeric_features].isna().mean(),
    'incoming_missing_rate': incoming[numeric_features].isna().mean(),
    #10 — Subtract means and divide by training SD; replace zero SD with NaN to avoid infinite but
    #     uninterpretable drift values.
    'standardized_mean_shift': (incoming[numeric_features].mean() - X_train[numeric_features].mean())
                               / X_train[numeric_features].std().replace(0, np.nan),
})
#11 — Flag an absolute missing-rate increase >.1 or a mean change >.5 training SD; | combines elementwise
#     Boolean signals.
monitoring['investigate'] = ((monitoring.incoming_missing_rate - monitoring.training_missing_rate) > 0.1
                             ) | (monitoring.standardized_mean_shift.abs() > 0.5)
#12 — Render this small named table locally; display does not save a file, fetch data, or establish external
#     approval.
display(monitoring)
#13 — The mean of Boolean review flags is review fraction, a workload indicator rather than a quality metric
#     without outcome labels.
print('Synthetic incoming action rate:', batch_response.review.mean())

### Try it — Part 17

Raise the synthetic shift and identify which signal fires first. Specify an incident owner and rollback condition; never auto-retrain from an unreviewed alert. When labels arrive, join them by approved identifiers and monitor outcome metrics and subgroup errors separately.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 17's evidence boundary. A copied validation batch is shifted without changing the original reference. Missing-rate changes, training-standardized mean shifts and review fraction expose different signals. Hypothesis: added missing tenure and raised spend trigger different monitors. Declared investigation thresholds are heuristics; ownership, rollback and delayed-label evidence remain required.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** This is a local batch function, not a server, container build or Kubernetes deployment. Feature drift may indicate collection issues or population changes, but cannot alone establish concept drift or lower outcome quality. Labels can arrive late, so operational and predictive monitoring have different clocks.

**Expected output and reading:** A copied validation batch is shifted without changing the original reference. Missing-rate changes, training-standardized mean shifts and review fraction expose different signals. Hypothesis: added missing tenure and raised spend trigger different monitors. Declared investigation thresholds are heuristics; ownership, rollback and delayed-label evidence remain required.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#3 — Fixed tuples pair each signal with an owner and first action; this is a rehearsal table, not a live
#     incident system.
runbook = pd.DataFrame([
    ('schema rejection', 'operator', 'reject batch; preserve aggregate diagnostic only'),
    ('feature drift', 'data steward', 'check collection change before considering retraining'),
    ('delayed-label quality drop', 'model owner', 'pause recommendations and use reviewed baseline'),
    ('privacy concern', 'privacy owner', 'stop processing and follow approved response procedure'),
], columns=['signal', 'owner', 'first action'])
#4 — Separate automatic test success from privacy and owner approvals that this notebook cannot grant.
release_gates = {'tests_pass': True, 'privacy_reviewed': False, 'owner_approved': False}
#5 — all requires every gate; false is intentionally correct and should not be changed just to make a demo
#     appear deployable.
print('Real release approved:', all(release_gates.values()), '(intentionally false)')
#6 — Render this small named table locally; display does not save a file, fetch data, or establish external
#     approval.
display(runbook)

<a id="part-18"></a>
# Part 18 — Teamwork and artifacts

[Read the textbook section](https://juandiegopleitez.com/MLstudy/teamwork.html)

**Learning objective:** Plan shared artifacts, acceptance criteria, blockers, and a credible handoff.

**Step 1.** Turn lifecycle responsibilities into a small team artifact board. Distinguish a passed notebook smoke test from privacy approval or measured intervention benefit. **Step 2.** Find blocked gates and identify the person who can resolve them. Nothing is written to a task service or repository.

## Guided lesson — Team artifacts are evidence gates

**Why this matters:** A model demo can establish reproducibility but cannot sign off a privacy review or prove intervention value. Each artifact needs an accountable owner, a checkable acceptance criterion and a status with a precise meaning. Blockers are dependencies, not a reason to relabel incomplete work as done.

**Expected inputs, outputs, and hypothesis:** A small artifact board becomes an open-work list and a two-audience handoff. Hypothesis: explicit evidence gates expose missing non-model work earlier than a single model-complete status. Rewrite one executive implication and one engineering boundary without weakening either limitation.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** A model demo can establish reproducibility but cannot sign off a privacy review or prove intervention value. Each artifact needs an accountable owner, a checkable acceptance criterion and a status with a precise meaning. Blockers are dependencies, not a reason to relabel incomplete work as done.

**Expected output and reading:** A small artifact board becomes an open-work list and a two-audience handoff. Hypothesis: explicit evidence gates expose missing non-model work earlier than a single model-complete status. Rewrite one executive implication and one engineering boundary without weakening either limitation.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#3 — Four-field tuples define artifact, owner, acceptance criterion and status; done applies only to stated
#     synthetic evidence.
artifact_board = pd.DataFrame([
    ('prediction contract', 'decision owner', 'unit, horizon, costs agreed', 'draft'),
    ('data card', 'data steward', 'synthetic provenance and schema checked', 'done'),
    ('model comparison', 'analyst', 'baseline and leakage-safe CV reported', 'done'),
    ('local inference contract', 'engineer', 'invalid inputs rejected; round-trip passes', 'done'),
    ('privacy review', 'privacy owner', 'appropriate review before any real data', 'blocked'),
    ('handoff', 'operator', 'runbook and accountable owner accepted', 'draft'),
], columns=['artifact', 'owner', 'acceptance criterion', 'status'])
#4 — Render this small named table locally; display does not save a file, fetch data, or establish external
#     approval.
display(artifact_board)
#5 — ne(done) builds a mask; loc selects incomplete artifact names and tolist makes a concise handoff list.
print('Open artifacts:', artifact_board.loc[artifact_board.status.ne('done'), 'artifact'].tolist())

### Try it — Part 18

Choose one draft artifact and write a checkable definition of done, not “improve model.” Draft an executive sentence about decision value and a technical sentence about the evidence boundary. Do not mark external reviews complete just because the notebook runs.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 18's evidence boundary. A small artifact board becomes an open-work list and a two-audience handoff. Hypothesis: explicit evidence gates expose missing non-model work earlier than a single model-complete status. Rewrite one executive implication and one engineering boundary without weakening either limitation.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** A model demo can establish reproducibility but cannot sign off a privacy review or prove intervention value. Each artifact needs an accountable owner, a checkable acceptance criterion and a status with a precise meaning. Blockers are dependencies, not a reason to relabel incomplete work as done.

**Expected output and reading:** A small artifact board becomes an open-work list and a two-audience handoff. Hypothesis: explicit evidence gates expose missing non-model work earlier than a single model-complete status. Rewrite one executive implication and one engineering boundary without weakening either limitation.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Named fields separate decision implications, technical boundaries, blockers and the next inspectable
#     demonstration.
handoff = {
    'decision_summary': 'Synthetic risk scores can prioritize a review queue, not prove outreach value.',
    'technical_summary': 'Training-fold preprocessing; validation-only cost threshold; one final holdout.',
    #2 — Derive blockers from the artifact board rather than maintaining an inconsistent second manual
    #     status list.
    'blocked_by': artifact_board.loc[artifact_board.status.eq('blocked'), 'artifact'].tolist(),
    'next_demo': 'review local batch rejection and aggregate drift alerts with the operator',
}
print(handoff)

<a id="part-19"></a>
# Part 19 — Applied projects

[Read the textbook section](https://juandiegopleitez.com/MLstudy/applied-projects.html)

**Learning objective:** Select a bounded capstone and turn a ten-day plan into measurable deliverables.

**Step 1.** Choose one of three synthetic capstones grounded in the book: retention triage, placeholder redaction, or sensor anomaly review. Existing demos provide baselines rather than empty TODOs. **Step 2.** Inspect the deliverable plan and add one new comparison with its evaluation boundary declared before running it. A ten-day schedule is illustrative, not a promise.

## Guided lesson — Bound one capstone before adding complexity

**Why this matters:** Choose a decision, baseline, output unit, evaluation boundary and finish line. A ten-day plan describes staged evidence, not a promise of institutional deployment. Customer scores, marker spans and anomaly alerts need different metrics and failure analyses.

**Expected inputs, outputs, and hypothesis:** A selected project yields its contract and an artifact schedule; an extra regularized tree yields training-CV balanced accuracy and AP. Hypothesis: larger minimum leaves trade local fit for stability. Because the previous final test has been used, a revised capstone needs a newly reserved evaluation set before any final superiority claim.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Choose a decision, baseline, output unit, evaluation boundary and finish line. A ten-day plan describes staged evidence, not a promise of institutional deployment. Customer scores, marker spans and anomaly alerts need different metrics and failure analyses.

**Expected output and reading:** A selected project yields its contract and an artifact schedule; an extra regularized tree yields training-CV balanced accuracy and AP. Hypothesis: larger minimum leaves trade local fit for stability. Because the previous final test has been used, a revised capstone needs a newly reserved evaluation set before any final superiority claim.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — Widget constructors specify frontend controls; guided_control runs the same default as ordinary Python first.
import ipywidgets as widgets
#3 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#4 — Outer keys name projects; nested dictionaries keep each question paired with its baseline, metric and
#     evaluation boundary.
capstones = {
    'retention': {'question': 'Who enters an optional review queue?', 'baseline': 'dummy vs logistic',
                  'metric': 'validation expected cost', 'boundary': 'customer train/validation; test already closed'},
    'redaction': {'question': 'Which synthetic marker spans are found?', 'baseline': 'literal marker rules',
                  'metric': 'exact-span precision and recall', 'boundary': 'new synthetic document templates'},
    'sensors': {'question': 'Which later readings need review?', 'baseline': 'forest vs reconstruction',
                'metric': 'AP plus alert precision', 'boundary': 'forward-time split; test already closed'},
}
#5 — Accept one known project key and display its contract/plan; unsupported keys fail rather than silently
#     selecting a different task.
def plan_capstone(project='retention'):
    #6 — Dictionary lookup selects one complete plan; changing this display does not retrain or deploy a
    #     model.
    print(capstones[project])
    #7 — Convert equally keyed rows or aligned vectors into a named table so comparisons retain their
    #     meaning and units.
    display(pd.DataFrame([
        ('1–2', 'contract and synthetic data audit'), ('3–4', 'baseline and split-safe preparation'),
        ('5–6', 'one controlled comparison'), ('7–8', 'error analysis and local interface'),
        ('9–10', 'limitations, runbook, demonstration and handoff'),
    ], columns=['days', 'evidence artifact']))
#8 — The helper executes the declared default, then registers callbacks; it returns no replacement final
#     model/policy. Dropdown options constrain valid values and value fixes a reproducible default.
guided_control(plan_capstone, project=widgets.Dropdown(options=list(capstones), value='retention', description='Project'))

### Try it — Part 19

Run this extra training-only comparison, then write an accept/reject decision relative to the dummy baseline using the same folds. Do not claim final test superiority: the original test has already been used, so a revised capstone needs a newly reserved evaluation set.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 19's evidence boundary. A selected project yields its contract and an artifact schedule; an extra regularized tree yields training-CV balanced accuracy and AP. Hypothesis: larger minimum leaves trade local fit for stability. Because the previous final test has been used, a revised capstone needs a newly reserved evaluation set before any final superiority claim.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Choose a decision, baseline, output unit, evaluation boundary and finish line. A ten-day plan describes staged evidence, not a promise of institutional deployment. Customer scores, marker spans and anomaly alerts need different metrics and failure analyses.

**Expected output and reading:** A selected project yields its contract and an artifact schedule; an extra regularized tree yields training-CV balanced accuracy and AP. Hypothesis: larger minimum leaves trade local fit for stability. Because the previous final test has been used, a revised capstone needs a newly reserved evaluation set before any final superiority claim.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — CV clones and fits the complete recipe separately inside each training fold.
from sklearn.model_selection import cross_validate
#2 — Classification trees learn split rules and leaf class proportions.
from sklearn.tree import DecisionTreeClassifier

#3 — A depth-3 tree with at least eight training rows per leaf is a declared complexity change, not an
#     arbitrary extra algorithm.
capstone_model = customer_model(DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, random_state=SEED))
#4 — Multiple scorer names evaluate the same fold fits; balanced accuracy tests hard decisions and AP tests
#     positive ranking.
capstone_scores = cross_validate(capstone_model, X_train, y_train, cv=cv,
                                 scoring=['balanced_accuracy', 'average_precision'])
#5 — Select test_ keys meaning CV-held-out fold results, not the closed final test; average each per-fold
#     vector.
print({metric: round(values.mean(), 3) for metric, values in capstone_scores.items()
       if metric.startswith('test_')})

<a id="part-20"></a>
# Part 20 — Glossary in practice

[Read the textbook section](https://juandiegopleitez.com/MLstudy/glossary.html)

**Learning objective:** Use precise vocabulary to explain actual calculations, not memorize labels.

**Step 1.** Pair glossary terms with operational definitions from the experiments. **Step 2.** Compute precision and recall from a miniature confusion matrix and distinguish a prediction threshold from a learned parameter, support from confidence, and drift from proven performance decay.

## Guided lesson — Use definitions to constrain interpretation

**Why this matters:** A parameter is learned state, a hyperparameter configures fitting, and a policy threshold maps scores to actions. Support, confidence, calibration and drift describe different evidence. Naming these correctly prevents accidental claims about causality, certainty or real-world deployment.

**Expected inputs, outputs, and hypothesis:** A glossary table links terms to previous code; fixed confusion counts become precision/recall/specificity/accuracy. Hypothesis: doubling false negatives leaves precision and specificity unchanged while reducing recall and accuracy. Derive this from denominators before executing.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** A parameter is learned state, a hyperparameter configures fitting, and a policy threshold maps scores to actions. Support, confidence, calibration and drift describe different evidence. Naming these correctly prevents accidental claims about causality, certainty or real-world deployment.

**Expected output and reading:** A glossary table links terms to previous code; fixed confusion counts become precision/recall/specificity/accuracy. Hypothesis: doubling false negatives leaves precision and specificity unchanged while reducing recall and accuracy. Derive this from denominators before executing.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#3 — Each row anchors a term to an operational definition and a concrete earlier code example.
glossary_examples = pd.DataFrame([
    ('parameter', 'coefficient fitted from training data', 'logistic coefficients'),
    ('hyperparameter', 'choice made outside parameter fitting', 'tree max_depth'),
    ('threshold', 'policy translating a score to an action', 'validation-selected cost cutoff'),
    ('leakage', 'unavailable or held-out information enters learning', 'fitting imputer before CV'),
    ('support', 'fraction of baskets containing the itemset', 'P(tea and bread)'),
    ('calibration', 'probability agreement with observed frequency', 'validation calibration bins'),
    ('drift', 'distribution changes that require investigation', 'incoming feature mean shift'),
], columns=['term', 'operational definition', 'notebook example'])
#4 — Render this small named table locally; display does not save a file, fetch data, or establish external
#     approval.
display(glossary_examples)
#5 — Use the same binary confusion order as Part 10; all metric denominators are nonzero in this constructed
#     example.
tn, fp, fn, tp = 18, 2, 3, 7
#6 — Precision divides TP by predicted positives, recall by true positives population, specificity by true
#     negatives population, accuracy by all rows.
print({'precision': tp / (tp + fp), 'recall': tp / (tp + fn),
       'specificity': tn / (tn + fp), 'accuracy': (tp + tn) / (tp + tn + fp + fn)})

### Try it — Part 20

Double the false negatives while keeping other counts fixed. Which metrics change, and why? Add a glossary example that distinguishes correlation from intervention effect; do not call association lift causal.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 20's evidence boundary. A glossary table links terms to previous code; fixed confusion counts become precision/recall/specificity/accuracy. Hypothesis: doubling false negatives leaves precision and specificity unchanged while reducing recall and accuracy. Derive this from denominators before executing.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** A parameter is learned state, a hyperparameter configures fitting, and a policy threshold maps scores to actions. Support, confidence, calibration and drift describe different evidence. Naming these correctly prevents accidental claims about causality, certainty or real-world deployment.

**Expected output and reading:** A glossary table links terms to previous code; fixed confusion counts become precision/recall/specificity/accuracy. Hypothesis: doubling false negatives leaves precision and specificity unchanged while reducing recall and accuracy. Derive this from denominators before executing.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Change only the missed-positive count in this arithmetic scenario; real threshold movement can change
#     several counts at once.
new_fn = 2 * fn
#2 — Precision ignores FN while recall/accuracy include it, explaining which ratios change under this
#     controlled count modification.
print({'precision_after': tp / (tp + fp), 'recall_after': tp / (tp + new_fn),
       'accuracy_after': (tp + tn) / (tp + tn + fp + new_fn)})
print('Correlation example: high support-ticket count predicts risk; reducing tickets is not proven to reduce churn.')

<a id="part-21"></a>
# Part 21 — Coverage and evidence boundaries

[Read the textbook section](https://juandiegopleitez.com/MLstudy/coverage.html)

**Learning objective:** Audit coverage while separating runnable demonstrations from conceptual references.

**Step 1.** Build a topic-to-evidence map. Every sidebar part has a runnable cell, but that does not mean every algorithm or infrastructure topic is implemented. **Step 2.** Inspect explicit limits: projections lose information, benchmark accuracy is not utility, and static downloads are not hosted execution.

## Guided lesson — Coverage is traceability, not mastery

**Why this matters:** Demonstrating a representative estimator does not evaluate the whole catalog. Infrastructure, compliance and scientific claims require independent evidence. A local interface is not a deployment; sklearn MLP is not TensorFlow; placeholder matching is not validated clinical redaction.

**Expected inputs, outputs, and hypothesis:** A topic/part/evidence-boundary table supports a gap audit; a structured next experiment names a minimal change, independent evaluation and success limit. Hypothesis: explicitly recording an unsupported capability produces a safer next step than claiming an entire adjacent topic is covered.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Demonstrating a representative estimator does not evaluate the whole catalog. Infrastructure, compliance and scientific claims require independent evidence. A local interface is not a deployment; sklearn MLP is not TensorFlow; placeholder matching is not validated clinical redaction.

**Expected output and reading:** A topic/part/evidence-boundary table supports a gap audit; a structured next experiment names a minimal change, independent evaluation and success limit. Hypothesis: explicitly recording an unsupported capability produces a safer next step than claiming an entire adjacent topic is covered.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas represents this local evidence/reference table; it does not train a model or validate an external claim.
import pandas as pd

#3 — Named fields keep each claim paired with its source, scope, and limitation.
coverage_rows = [
    ('classifier/regressor', '01, 04–08, 10', 'logistic, tree, SVC, Ridge, SVR; not the whole catalog'),
    ('clustering/reduction', '09', 'k-means, hierarchy, mixture, PCA; exploratory geometry'),
    ('text/ensembles', '11–12', 'tiny original corpus and baskets; no pretrained NER or causal claims'),
    ('neural/anomaly', '13', 'sklearn MLP and isolation forest; not TensorFlow'),
    ('optimization', '14', 'continuous LP and quadratic; not an integer solver tutorial'),
    ('PII/PHI', '15', 'synthetic placeholder spans only; not compliant de-identification'),
    ('engineering/delivery', '16–17', 'local contract and temporary bundle; no real deployment'),
    ('team/reference', '18–22', 'artifact plans, checks and evidence; no institutional approval'),
]
#4 — DataFrame columns distinguish the topic, supporting parts and claim boundary; this is traceability, not an algorithm benchmark.
coverage_map = pd.DataFrame(coverage_rows, columns=['topic', 'parts', 'evidence boundary'])
display(coverage_map)
#5 — The eight-row invariant tests only that all declared topic groups were recorded, not that the reader mastered them.
assert len(coverage_map) == 8
print('Coverage is traceability, not mastery or certification.')

### Try it — Part 21

Choose one unimplemented topic and specify the smallest safe next experiment, an evaluation set, and a success criterion. For deployment topics, keep the experiment local and avoid publishing endpoints or using real personal records.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 21's evidence boundary. A topic/part/evidence-boundary table supports a gap audit; a structured next experiment names a minimal change, independent evaluation and success limit. Hypothesis: explicitly recording an unsupported capability produces a safer next step than claiming an entire adjacent topic is covered.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Demonstrating a representative estimator does not evaluate the whole catalog. Infrastructure, compliance and scientific claims require independent evidence. A local interface is not a deployment; sklearn MLP is not TensorFlow; placeholder matching is not validated clinical redaction.

**Expected output and reading:** A topic/part/evidence-boundary table supports a gap audit; a structured next experiment names a minimal change, independent evaluation and success limit. Hypothesis: explicitly recording an unsupported capability produces a safer next step than claiming an entire adjacent topic is covered.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — A structured gap plan states the smallest change, independent evaluation, success criterion and
#     remaining limitation.
next_experiment = {
    'gap': 'robust placeholder span detection across synthetic templates',
    'smallest_change': 'add NAME markers and varied punctuation',
    'evaluation': 'reserve new templates before extending the pattern',
    'criterion': 'report exact-span false positives and false negatives, not accuracy alone',
    'limits': 'still synthetic; no claim of real PHI coverage or compliance',
}
print(next_experiment)

<a id="part-22"></a>
# Part 22 — Source notes and reference checks

[Read the textbook section](https://juandiegopleitez.com/MLstudy/source-notes.html)

**Learning objective:** Reconcile reference notes with tested calculations and keep claims traceable.

**Step 1.** Use the source-notes page as provenance and a place to revisit original formulations; use the textbook’s corrections and the experiments as checks, not as authority without evidence. **Step 2.** Build a small claim ledger that distinguishes formulas verified here from external scientific claims. Public documentation links are reading references only; no network requests occur.

## Guided lesson — References should lead to checkable claims

**Why this matters:** Source notes preserve provenance but do not make every formulation authoritative. Compare textbook corrections, local experiments and package behavior under recorded versions. Executable arithmetic can test an assumption-bound claim, not an unmeasured institutional or causal effect.

**Expected inputs, outputs, and hypothesis:** A claim ledger separates verified mechanics from unsupported generalization. Public URL strings are reading aids, not runtime network calls. Final checks verify formulas, LP profit, synthetic span offsets and full CSV/fallback equivalence. Hypothesis: naming the evidence boundary is more useful than adding citations to an unsupported conclusion.

**Prerequisite boundary:** run all preceding learning cells. The local imports below make requirements visible, but earlier schemas, helpers, split decisions and trained objects are deliberately reused. Read a constructor as configuration, fit as learning, transform as application of learned representation, and metrics as evaluation—not interchangeable operations.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Source notes preserve provenance but do not make every formulation authoritative. Compare textbook corrections, local experiments and package behavior under recorded versions. Executable arithmetic can test an assumption-bound claim, not an unmeasured institutional or causal effect.

**Expected output and reading:** A claim ledger separates verified mechanics from unsupported generalization. Public URL strings are reading aids, not runtime network calls. Final checks verify formulas, LP profit, synthetic span offsets and full CSV/fallback equivalence. Hypothesis: naming the evidence boundary is more useful than adding citations to an unsupported conclusion.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — display renders the local table/control representation without saving an artifact or contacting a service.
from IPython.display import display
#2 — pandas represents this local evidence/reference table; it does not train a model or validate an external claim.
import pandas as pd

#3 — Named fields keep each claim paired with its source, scope, and limitation.
claim_ledger = pd.DataFrame([
    ('Bayes posterior depends on prevalence', 'Part 05; rare-alert arithmetic', 'verified under stated assumptions'),
    ('Lift differs from causal effect', 'Part 12; descriptive basket frequencies', 'causal effect not estimated'),
    ('Preprocessing belongs inside CV', 'Parts 04 and 08; unfitted pipelines per fold', 'implementation boundary inspected'),
    ('Input contracts reject malformed batches', 'Part 16; negative smoke tests', 'only tested contract covered'),
    ('Local model generalizes to a real institution', 'no real-world evaluation', 'unsupported'),
], columns=['claim', 'evidence', 'status'])
#4 — Inspect evidence and status together: a passed local arithmetic check cannot support an unmeasured institutional generalization.
display(claim_ledger)
references = {
    'textbook source notes': 'https://juandiegopleitez.com/MLstudy/source-notes.html',
    'sklearn user guide': 'https://scikit-learn.org/stable/user_guide.html',
    'scipy optimization': 'https://docs.scipy.org/doc/scipy/reference/optimize.html',
    'ipywidgets': 'https://ipywidgets.readthedocs.io/en/stable/',
}
print(references)

### Try it — Part 22

Pick a formula from the source notes and reproduce it on a tiny constructed example without copying source prose. Record assumptions, a counterexample or limit, and the package version. Finish by restarting the kernel and running all cells; optional CSV and standalone modes should have identical input schemas and values.

**Work the exercise deliberately:** before running, state one concrete hypothesis and name the expected input/output representation. After running, compare with Part 22's evidence boundary. A claim ledger separates verified mechanics from unsupported generalization. Public URL strings are reading aids, not runtime network calls. Final checks verify formulas, LP profit, synthetic span offsets and full CSV/fallback equivalence. Hypothesis: naming the evidence boundary is more useful than adding citations to an unsupported conclusion.

Record what this calculation cannot establish and which independent observations would be needed next. The following cell is a working starting point, not an empty TODO.

### Code walkthrough

**Expected input:** the state built by all preceding learning cells in this notebook. Importing a utility here does not recreate datasets or fitted models.

**Procedure:** Source notes preserve provenance but do not make every formulation authoritative. Compare textbook corrections, local experiments and package behavior under recorded versions. Executable arithmetic can test an assumption-bound claim, not an unmeasured institutional or causal effect.

**Expected output and reading:** A claim ledger separates verified mechanics from unsupported generalization. Public URL strings are reading aids, not runtime network calls. Final checks verify formulas, LP profit, synthetic span offsets and full CSV/fallback equivalence. Hypothesis: naming the evidence boundary is more useful than adding citations to an unsupported conclusion.

Before executing, identify which values in this cell are declared assumptions, which are fitted state, and which are evaluation results. After executing, explain a result that would contradict the stated hypothesis; do not merely repeat a printed score.

In [ ]:
#1 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import numpy as np
#2 — Import this local dependency before using its API; the following numbered notes explain the learned state and input/output contract.
import pandas as pd

#3 — Named scalar checks make an assertion failure point to its assumption instead of just an unlabeled
#     Boolean.
checks = {
    #4 — isclose allows numerical tolerance for logs and optimization, avoiding brittle exact floating-point
    #     equality.
    'entropy_of_fair_binary': np.isclose(entropy([0.5, 0.5]), 1),
    #5 — Constructed independent marginals .5 and joint .25 imply lift one; this is not an estimate that the
    #     observed baskets are independent.
    'independent_basket_lift': np.isclose(0.25 / (0.5 * 0.5), 1),
    #6 — Recompute positive profit from the original verified LP, not any capacity-widget scenario output.
    'LP_verified_profit': np.isclose(profit @ production.x, 9),
    #7 — Whole-set equality verifies both exact boundaries and labels on the fixed synthetic note.
    'span_offsets_correct': predicted_spans == gold_spans,
    #8 — Ordered column-name equality checks representation; the full-frame assertions additionally check
    #     every value and missing cell.
    'customer_schema': list(customers.columns) == list(synthetic_customers().columns),
    'sensor_schema': list(sensors.columns) == list(synthetic_sensors().columns),
}
#9 — Combine named invariants and include the dictionary as failure context; passing is a scoped self-check,
#     not certification.
assert all(checks.values()), checks
#10 — Compare loaded/fallback customer data with a fresh generator output; permit equivalent inferred dtypes
#     only.
pd.testing.assert_frame_equal(customers, synthetic_customers(), check_dtype=False)
#11 — Apply the same schema/order/value equivalence test to sensor data, proving optional CSV and standalone
#     modes match.
pd.testing.assert_frame_equal(sensors, synthetic_sensors(), check_dtype=False)
print('Final self-checks:', checks)
print('Completed 22 guided parts. No network, deployment, or persistent data/model writes were needed.')